# Coursework 1

In [ ]:
!pip install pandas openpyxl altair

In [ ]:
import pandas as pd
import altair as alt

### Load and Combine

In [ ]:
file_path = r"C:\Users\bilal\Downloads\CST4245 CW1\data\cw1 -dataset.xlsx"

excel_file = pd.ExcelFile(file_path)
print(excel_file.sheet_names)

In [ ]:
bp = pd.read_excel(file_path, sheet_name="Raised Blood Pressure")
bmi = pd.read_excel(file_path, sheet_name="BMI")
diabetes = pd.read_excel(file_path, sheet_name="Diabetes")

In [ ]:
print(bp.head())
print(bmi.head())
print(diabetes.head())

In [ ]:
bp = bp.rename(columns={"Prevalence of raised blood pressure": "Prevalence"})
bmi = bmi.rename(columns={"Prevalence of BMI>=30 kg/m≤ (obesity)": "Prevalence"})
diabetes = diabetes.rename(columns={"Age-standardised diabetes prevalence": "Prevalence"})

In [ ]:
bp["Indicator"] = "Raised Blood Pressure"
bmi["Indicator"] = "Obesity"
diabetes["Indicator"] = "Diabetes"

In [ ]:
master = pd.concat([bp, bmi, diabetes], ignore_index=True)

In [ ]:
print(master.head())
print(master.shape)
print(master["Indicator"].value_counts())

In [ ]:
master["PrevalencePercent"] = master["Prevalence"] * 100

In [ ]:
master.to_csv("C:/Users/bilal/Downloads/CST4245 CW1/outputs/master health dataset.csv", index=False)

### World Bank GDP file

In [ ]:
gdp = pd.read_csv("C:/Users/bilal/Downloads/CST4245 CW1/data/GDP.csv", skiprows=4)
gdp.head()

### Keep relevant columns and convert it from wide format to long format

In [ ]:
gdp = gdp.drop(columns=["Country Code", "Indicator Name", "Indicator Code", "Unnamed: 70"], errors="ignore")

In [ ]:
gdp_long = gdp.melt(
    id_vars=["Country Name"],
    var_name="Year",
    value_name="GDPPerCapita"
)

### Keep relvant years and match the countries with the main dataset

In [ ]:
gdp_long = gdp_long[gdp_long["Year"].str.isnumeric()].copy()
gdp_long["Year"] = gdp_long["Year"].astype(int)
gdp_long = gdp_long[(gdp_long["Year"] >= 1975) & (gdp_long["Year"] <= 2016)]

In [ ]:
gdp_long = gdp_long.rename(columns={"Country Name": "Country/Region/World"})

### Map the countries

In [ ]:
master_countries = set(master["Country/Region/World"].unique())
gdp_countries = set(gdp_long["Country/Region/World"].unique())

only_in_master = sorted(master_countries - gdp_countries)
only_in_gdp = sorted(gdp_countries - master_countries)

print("Only in master:", only_in_master[:20])
print("Only in GDP:", only_in_gdp[:20])

In [ ]:
country_fixes = {
    "Egypt": "Egypt, Arab Rep.",
    "Iran": "Iran, Islamic Rep.",
    "Turkey": "Turkiye",
    "Venezuela": "Venezuela, RB",
    "Yemen": "Yemen, Rep.",
    "Syria": "Syrian Arab Republic",
    "Russia": "Russian Federation",
    "Slovakia": "Slovak Republic",
    "Kyrgyzstan": "Kyrgyz Republic",
    "Czech Republic": "Czechia",
    "South Korea": "Korea, Rep.",
    "North Korea": "Korea, Dem. People's Rep.",
    "Laos": "Lao PDR",
    "Gambia": "Gambia, The",
    "Bahamas": "Bahamas, The",
    "Congo": "Congo, Rep.",
    "Democratic Republic of the Congo": "Congo, Dem. Rep.",
    "Micronesia": "Micronesia, Fed. Sts.",
    "Cape Verde": "Cabo Verde",
    "Brunei": "Brunei Darussalam"
}

master["Country/Region/World"] = master["Country/Region/World"].replace(country_fixes)

In [ ]:
master_countries = set(master["Country/Region/World"].unique())
gdp_countries = set(gdp_long["Country/Region/World"].unique())

print("Still unmatched in master:", sorted(master_countries - gdp_countries)[:30])

In [ ]:
country_fixes_more = {
    "China (Hong Kong SAR)": "Hong Kong SAR, China",
    "DR Congo": "Congo, Dem. Rep.",
    "Guinea Bissau": "Guinea-Bissau",
    "Macedonia (TFYR)": "North Macedonia",
    "Micronesia (Federated States of)": "Micronesia, Fed. Sts.",
    "Occupied Palestinian Territory": "West Bank and Gaza",
    "Saint Kitts and Nevis": "St. Kitts and Nevis",
    "Saint Lucia": "St. Lucia",
    "Saint Vincent and the Grenadines": "St. Vincent and the Grenadines",
    "Swaziland": "Eswatini",
    "United States of America": "United States"
}

master["Country/Region/World"] = master["Country/Region/World"].replace(country_fixes_more)

In [ ]:
master_countries = set(master["Country/Region/World"].unique())
gdp_countries = set(gdp_long["Country/Region/World"].unique())

print("Still unmatched in master:", sorted(master_countries - gdp_countries))

### Merge the datasets

In [ ]:
master = master.merge(
    gdp_long,
    on=["Country/Region/World", "Year"],
    how="left"
)

In [ ]:
print(master[["Country/Region/World", "Year", "Indicator", "GDPPerCapita"]].head())
print(master["GDPPerCapita"].isna().mean())

In [ ]:
master.to_csv("C:/Users/bilal/Downloads/CST4245 CW1/outputs/master with gdp.csv", index=False)

### Health Expenditure Dataset, same process followed

In [ ]:
health_exp = pd.read_csv("C:/Users/bilal/Downloads/CST4245 CW1/data/Health Expenditure.csv", skiprows=4)
health_exp.head()

In [ ]:
health_exp = health_exp.drop(
    columns=["Country Code", "Indicator Name", "Indicator Code", "Unnamed: 70"],
    errors="ignore"
)

In [ ]:
health_exp_long = health_exp.melt(
    id_vars=["Country Name"],
    var_name="Year",
    value_name="HealthExpenditurePerCapita"
)

In [ ]:
health_exp_long = health_exp_long[health_exp_long["Year"].str.isnumeric()].copy()
health_exp_long["Year"] = health_exp_long["Year"].astype(int)

health_exp_long = health_exp_long[
    (health_exp_long["Year"] >= 1975) &
    (health_exp_long["Year"] <= 2016)
]

In [ ]:
health_exp_long = health_exp_long.rename(
    columns={"Country Name": "Country/Region/World"}
)

In [ ]:
health_exp_long["Country/Region/World"] = health_exp_long["Country/Region/World"].replace(country_fixes)
health_exp_long["Country/Region/World"] = health_exp_long["Country/Region/World"].replace(country_fixes_more)

In [ ]:
master_countries = set(master["Country/Region/World"].unique())
health_countries = set(health_exp_long["Country/Region/World"].unique())

print("Still unmatched:", sorted(master_countries - health_countries))

In [ ]:
master = master.merge(
    health_exp_long,
    on=["Country/Region/World", "Year"],
    how="left"
)

In [ ]:
print(master[[
    "Country/Region/World",
    "Year",
    "Indicator",
    "GDPPerCapita",
    "HealthExpenditurePerCapita"
]].head())

print(master["HealthExpenditurePerCapita"].isna().mean())

In [ ]:
health_exp_long["Year"].min(), health_exp_long["Year"].max()

In [ ]:
master.to_csv("C:/Users/bilal/Downloads/CST4245 CW1/outputs/master_with_gdp_health.csv", index=False)

### Urban Population Dataset, same process followed

In [ ]:
urban = pd.read_csv("C:/Users/bilal/Downloads/CST4245 CW1/data/Urban Population.csv", skiprows=4)
urban.head()

In [ ]:
urban = urban.drop(
    columns=["Country Code", "Indicator Name", "Indicator Code", "Unnamed: 70"],
    errors="ignore"
)

In [ ]:
urban_long = urban.melt(
    id_vars=["Country Name"],
    var_name="Year",
    value_name="UrbanPopulationPercent"
)

In [ ]:
urban_long = urban_long[urban_long["Year"].str.isnumeric()].copy()
urban_long["Year"] = urban_long["Year"].astype(int)

urban_long = urban_long[
    (urban_long["Year"] >= 1975) &
    (urban_long["Year"] <= 2016)
]

In [ ]:
urban_long = urban_long.rename(
    columns={"Country Name": "Country/Region/World"}
)

In [ ]:
urban_long["Country/Region/World"] = urban_long["Country/Region/World"].replace(country_fixes)
urban_long["Country/Region/World"] = urban_long["Country/Region/World"].replace(country_fixes_more)

In [ ]:
master = master.merge(
    urban_long,
    on=["Country/Region/World", "Year"],
    how="left"
)

In [ ]:
print(master["UrbanPopulationPercent"].isna().mean())

In [ ]:
master.to_csv("C:/Users/bilal/Downloads/CST4245 CW1/outputs/master_with_gdp_health_urban.csv", index=False)

### Age 65 Dataset, same process followed

In [ ]:
age65 = pd.read_csv("C:/Users/bilal/Downloads/CST4245 CW1/data/65 and Above.csv", skiprows=4)
age65.head()

In [ ]:
age65 = age65.drop(
    columns=["Country Code", "Indicator Name", "Indicator Code", "Unnamed: 70"],
    errors="ignore"
)

In [ ]:
age65_long = age65.melt(
    id_vars=["Country Name"],
    var_name="Year",
    value_name="Age65PlusPercent"
)

In [ ]:
age65_long = age65_long[age65_long["Year"].str.isnumeric()].copy()
age65_long["Year"] = age65_long["Year"].astype(int)

age65_long = age65_long[
    (age65_long["Year"] >= 1975) &
    (age65_long["Year"] <= 2016)
]

In [ ]:
age65_long = age65_long.rename(
    columns={"Country Name": "Country/Region/World"}
)

In [ ]:
age65_long["Country/Region/World"] = age65_long["Country/Region/World"].replace(country_fixes)
age65_long["Country/Region/World"] = age65_long["Country/Region/World"].replace(country_fixes_more)

In [ ]:
master = master.merge(
    age65_long,
    on=["Country/Region/World", "Year"],
    how="left"
)

In [ ]:
print(master["Age65PlusPercent"].isna().mean())

In [ ]:
master.to_csv("C:/Users/bilal/Downloads/CST4245 CW1/outputs/final master dataset.csv", index=False)

### Income Classification Dataset

In [ ]:
income = pd.read_excel("C:/Users/bilal/Downloads/CST4245 CW1/data/CLASS_2025_10_07.xlsx")
income.head()

In [ ]:
income = income[["Economy", "Region", "Income group"]]

### Retain only necessary columns

In [ ]:
income = income.rename(columns={
    "Economy": "Country/Region/World",
    "Region": "Region",
    "Income group": "IncomeGroup"
})

In [ ]:
income["Country/Region/World"] = income["Country/Region/World"].replace(country_fixes)
income["Country/Region/World"] = income["Country/Region/World"].replace(country_fixes_more)

In [ ]:
master = master.merge(
    income,
    on="Country/Region/World",
    how="left"
)

In [ ]:
print(master[["Country/Region/World", "Region", "IncomeGroup"]].head())
print(master["IncomeGroup"].isna().mean())

In [ ]:
print(master["Region"].unique())
print(master["Region"].isna().mean())

In [ ]:
master["Region"] = master["Region"].fillna("Other")

In [ ]:
print(master["Region"].isna().mean())

In [ ]:
master.to_csv("C:/Users/bilal/Downloads/CST4245 CW1/outputs/final master dataset complete.csv", index=False)

# Graphs and Charts

### Visual 1
Global Trend of Cardiometabolic Risk Indicators

In [ ]:
import altair as alt
import pandas as pd

alt.data_transformers.disable_max_rows()

# Color mapping to match your legend
indicator_colors = {
    "Diabetes": "#4C78A8",
    "Obesity": "#F58518",
    "Raised Blood Pressure": "#E45756"
}

# Main line chart without legend
chart1 = alt.Chart(master).mark_line(point=True).encode(
    x=alt.X('Year:O', title='Year'),
    y=alt.Y('mean(PrevalencePercent):Q', title='Average Prevalence (%)'),
    color=alt.Color(
        'Indicator:N',
        scale=alt.Scale(
            domain=list(indicator_colors.keys()),
            range=list(indicator_colors.values())
        ),
        legend=None
    ),
    tooltip=[
        'Indicator',
        'Year',
        alt.Tooltip('mean(PrevalencePercent):Q', title='Average Prevalence', format='.2f')
    ]
).properties(
    width=700,
    height=400,
    title="Global Trend of Cardiometabolic Risk Indicators (1975 to 2016)"
)

# Create summary data for cards
card_data = (
    master.groupby("Indicator", as_index=False)["PrevalencePercent"]
    .mean()
    .rename(columns={"PrevalencePercent": "AveragePrevalence"})
)

card_data["Color"] = card_data["Indicator"].map(indicator_colors)
card_data["Label"] = card_data["AveragePrevalence"].round(2).astype(str) + "%"

# Base for cards
card_base = alt.Chart(card_data).encode(
    y=alt.Y('Indicator:N', sort=list(indicator_colors.keys()), axis=None)
)

# Colored rectangle cards
card_rect = card_base.mark_rect(
    cornerRadius=12,
    height=85
).encode(
    color=alt.Color('Color:N', scale=None)
).properties(
    width=180,
    height=300
)

# Indicator name text
card_title = card_base.mark_text(
    align='center',
    baseline='middle',
    fontSize=15,
    fontWeight='bold',
    color='white',
    dy=-12
).encode(
    text='Indicator:N'
)

# Average value text
card_value = card_base.mark_text(
    align='center',
    baseline='middle',
    fontSize=20,
    fontWeight='bold',
    color='white',
    dy=16
).encode(
    text='Label:N'
)

cards = (card_rect + card_title + card_value).properties(
    title="Average by Indicator"
)

# Combine line chart and cards
final_chart1 = alt.hconcat(
    chart1,
    cards,
    spacing=20
).resolve_scale(
    color='independent'
).configure_view(
    stroke=None
)

final_chart1

### Visual 2
Sex Differences Across Indicators

In [ ]:
import altair as alt
import pandas as pd

alt.data_transformers.disable_max_rows()

sex_summary = (
    master.groupby(["Year", "Indicator", "Sex"], as_index=False)["PrevalencePercent"]
    .mean()
)

indicator_options = sorted(sex_summary["Indicator"].unique())

indicator_dropdown = alt.param(
    name="Select",
    bind=alt.binding_select(options=indicator_options, name="Indicator: "),
    value=indicator_options[0]
)

sex_colors = {
    "Men": "#4C78A8",
    "Women": "#E45756"
}

# Main line chart without legend
chart2 = (
    alt.Chart(sex_summary)
    .add_params(indicator_dropdown)
    .transform_filter(alt.datum.Indicator == indicator_dropdown)
    .mark_line(point=True, strokeWidth=3)
    .encode(
        x=alt.X("Year:O", title="Year"),
        y=alt.Y(
            "PrevalencePercent:Q",
            title="Average Prevalence (%)",
            scale=alt.Scale(zero=False)
        ),
        color=alt.Color(
            "Sex:N",
            scale=alt.Scale(
                domain=list(sex_colors.keys()),
                range=list(sex_colors.values())
            ),
            legend=None
        ),
        tooltip=[
            alt.Tooltip("Indicator:N", title="Indicator"),
            alt.Tooltip("Sex:N", title="Sex"),
            alt.Tooltip("Year:O", title="Year"),
            alt.Tooltip("PrevalencePercent:Q", title="Average Prevalence (%)", format=".2f")
        ]
    )
    .properties(
        width=700,
        height=400,
        title="Sex Differences by Selected Cardiometabolic Indicator"
    )
)

# Summary cards data that responds to selected indicator
sex_card_data = (
    alt.Chart(sex_summary)
    .add_params(indicator_dropdown)
    .transform_filter(alt.datum.Indicator == indicator_dropdown)
    .transform_aggregate(
        AveragePrevalence="mean(PrevalencePercent)",
        groupby=["Sex"]
    )
    .transform_calculate(
        Color=(
            "datum.Sex == 'Men' ? '#4C78A8' : '#E45756'"
        ),
        Label="format(datum.AveragePrevalence, '.2f') + '%'"
    )
)

# Card rectangles
card_rect = sex_card_data.mark_rect(
    cornerRadius=12,
    height=95
).encode(
    y=alt.Y(
        "Sex:N",
        sort=["Men", "Women"],
        axis=None,
        scale=alt.Scale(paddingInner=0.18, paddingOuter=0.12)
    ),
    color=alt.Color("Color:N", scale=None)
).properties(
    width=180,
    height=260
)

# Card title text
card_title = sex_card_data.mark_text(
    align="center",
    baseline="middle",
    fontSize=18,
    fontWeight="bold",
    color="white",
    dy=-14
).encode(
    y=alt.Y(
        "Sex:N",
        sort=["Men", "Women"],
        axis=None,
        scale=alt.Scale(paddingInner=0.18, paddingOuter=0.12)
    ),
    text="Sex:N"
)

# Card value text
card_value = sex_card_data.mark_text(
    align="center",
    baseline="middle",
    fontSize=24,
    fontWeight="bold",
    color="white",
    dy=18
).encode(
    y=alt.Y(
        "Sex:N",
        sort=["Men", "Women"],
        axis=None,
        scale=alt.Scale(paddingInner=0.15, paddingOuter=0.12)
    ),
    text="Label:N"
)

cards = (card_rect + card_title + card_value).properties(
    title=alt.TitleParams(
        text="Average by Sex",
        anchor="middle"
    )
)

final_chart2 = alt.hconcat(
    chart2,
    cards,
    spacing=20
).resolve_scale(
    color="independent"
).configure_title(
    fontSize=18,
    anchor="start"
).configure_axis(
    labelFontSize=11,
    titleFontSize=13
).configure_view(
    stroke=None
)

final_chart2

### Visual 3
Regional Trends with Interactive Indicator Filter

In [ ]:
region_summary = (
    master.dropna(subset=["Region"])
    .groupby(["Year", "Indicator", "Region"], as_index=False)["PrevalencePercent"]
    .mean()
)

region_options = sorted(region_summary["Indicator"].unique())

region_indicator_dropdown = alt.param(
    name="SelectIndicator",
    bind=alt.binding_select(options=region_options, name="Indicator: "),
    value=region_options[0]
)

chart3 = (
    alt.Chart(region_summary)
    .add_params(region_indicator_dropdown)
    .transform_filter(alt.datum.Indicator == region_indicator_dropdown)
    .mark_line(point=True, strokeWidth=2.5)
    .encode(
        x=alt.X("Year:O", title="Year"),
        y=alt.Y(
            "PrevalencePercent:Q",
            title="Average Prevalence (%)",
            scale=alt.Scale(zero=False)
        ),
        color=alt.Color("Region:N", title="Region"),
        tooltip=[
            alt.Tooltip("Region:N", title="Region"),
            alt.Tooltip("Indicator:N", title="Indicator"),
            alt.Tooltip("Year:Q", title="Year"),
            alt.Tooltip("PrevalencePercent:Q", title="Average Prevalence (%)", format=".2f")
        ]
    )
    .properties(
        width=700,
        height=400,
        title="Regional Trends in Cardiometabolic Risk"
    )
    .configure_title(fontSize=18, anchor="start")
    .configure_axis(labelFontSize=11, titleFontSize=13)
    .configure_legend(titleFontSize=12, labelFontSize=11)
)

chart3

In [ ]:
region_summary = (
    master.dropna(subset=["Region"])
    .groupby(["Year", "Indicator", "Region"], as_index=False)["PrevalencePercent"]
    .mean()
)

region_summary = region_summary[region_summary["Region"] != "Other"]

region_options = sorted(region_summary["Indicator"].unique())

region_indicator_dropdown = alt.param(
    name="SelectIndicator",
    bind=alt.binding_select(options=region_options, name="Indicator: "),
    value=region_options[0]
)

region_order = [
    "North America",
    "Middle East, North Africa, Afghanistan & Pakistan",
    "East Asia & Pacific",
    "Europe & Central Asia",
    "Latin America & Caribbean",
    "Sub-Saharan Africa",
    "South Asia"
]

base = (
    alt.Chart(region_summary)
    .add_params(region_indicator_dropdown)
    .transform_filter(alt.datum.Indicator == region_indicator_dropdown)
)

heatmap = base.mark_rect().encode(
    x=alt.X("Year:O", title="Year"),
    y=alt.Y("Region:N", title="Region", sort=region_order),
    color=alt.Color(
        "PrevalencePercent:Q",
        title="Average Prevalence (%)",
        scale=alt.Scale(scheme="redyellowgreen", reverse=True)
    ),
    tooltip=[
        alt.Tooltip("Region:N", title="Region"),
        alt.Tooltip("Indicator:N", title="Indicator"),
        alt.Tooltip("Year:O", title="Year"),
        alt.Tooltip("PrevalencePercent:Q", title="Average Prevalence (%)", format=".0f")
    ]
)

text = base.mark_text(size=9).encode(
    x=alt.X("Year:O"),
    y=alt.Y("Region:N", sort=region_order),
    text=alt.Text("PrevalencePercent:Q", format=".0f"),
    color=alt.value("black")
)
chart3 = (heatmap + text).properties(
    width=900,
    height=320,
    title="Regional Heatmap of Cardiometabolic Risk"
).configure_title(
    fontSize=18,
    anchor="start"
).configure_axis(
    labelFontSize=11,
    titleFontSize=13
).configure_legend(
    titleFontSize=12,
    labelFontSize=11
)

chart3

### Visual 4
GDP per Capita vs Health Risk

In [ ]:
gdp_data = master.dropna(subset=["GDPPerCapita"])

In [ ]:
indicator_options2 = sorted(gdp_data["Indicator"].unique())

indicator_dropdown = alt.param(
    name="SelectIndicator",
    bind=alt.binding_select(options=indicator_options2, name="Indicator: "),
    value=indicator_options2[0]
)

chart4 = (
    alt.Chart(gdp_data)
    .add_params(indicator_dropdown)
    .transform_filter(alt.datum.Indicator == indicator_dropdown)
    .mark_circle(size=70, opacity=0.6)
    .encode(
        x=alt.X(
            "GDPPerCapita:Q",
            title="GDP per Capita"
        ),
        y=alt.Y(
            "PrevalencePercent:Q",
            title="Prevalence (%)"
        ),
        color=alt.Color(
            "Region:N",
            title="Region"
        ),
        tooltip=[
            alt.Tooltip("Country/Region/World:N", title="Country"),
            alt.Tooltip("Region:N", title="Region"),
            alt.Tooltip("Year:Q", title="Year"),
            alt.Tooltip("GDPPerCapita:Q", title="GDP per Capita", format=",.0f"),
            alt.Tooltip("PrevalencePercent:Q", title="Prevalence (%)", format=".1f")
        ]
    )
    .properties(
        width=700,
        height=400,
        title="Relationship Between Economic Development and Cardiometabolic Risk"
    )
    .configure_title(
        fontSize=18,
        anchor="start"
    )
)

chart4

In [ ]:
import altair as alt
import pandas as pd

linked_data = master.dropna(subset=["GDPPerCapita", "PrevalencePercent", "Region"]).copy()
linked_data = linked_data[linked_data["Region"] != "Other"]

# Aggregate across sex so each country-year-indicator has one row
linked_data_agg = (
    linked_data.groupby(
        ["Country/Region/World", "Region", "Year", "Indicator"],
        as_index=False
    )
    .agg(
        GDPPerCapita=("GDPPerCapita", "mean"),
        PrevalencePercent=("PrevalencePercent", "mean")
    )
)

indicator_options3 = sorted(linked_data_agg["Indicator"].unique())

indicator_dropdown2 = alt.param(
    name="SelectIndicator2",
    bind=alt.binding_select(options=indicator_options3, name="Indicator: "),
    value=indicator_options3[0]
)

year_slider2 = alt.param(
    name="SelectYear2",
    bind=alt.binding_range(
        min=int(linked_data_agg["Year"].min()),
        max=int(linked_data_agg["Year"].max()),
        step=1,
        name="Year: "
    ),
    value=2010
)

brush2 = alt.selection_interval(name="ScatterBrush2")

base2 = (
    alt.Chart(linked_data_agg)
    .add_params(indicator_dropdown2, year_slider2, brush2)
    .transform_filter(alt.datum.Indicator == indicator_dropdown2)
    .transform_filter(alt.datum.Year == year_slider2)
)

scatter2 = base2.mark_circle(size=90).encode(
    x=alt.X(
        "GDPPerCapita:Q",
        scale=alt.Scale(type="log"),
        title="GDP per Capita (log scale)"
    ),
    y=alt.Y(
        "PrevalencePercent:Q",
        title="Prevalence (%)"
    ),
    color=alt.condition(
        brush2,
        alt.Color("Region:N", title="Region"),
        alt.value("lightgray")
    ),
    tooltip=[
        alt.Tooltip("Country/Region/World:N", title="Country"),
        alt.Tooltip("Region:N", title="Region"),
        alt.Tooltip("Year:Q", title="Year"),
        alt.Tooltip("GDPPerCapita:Q", title="GDP per Capita", format=",.0f"),
        alt.Tooltip("PrevalencePercent:Q", title="Prevalence (%)", format=".1f")
    ]
).properties(
    width=520,
    height=420,
)

bars2 = (
    base2.transform_filter(brush2)
    .transform_aggregate(
        AvgPrevalence="mean(PrevalencePercent)",
        groupby=["Country/Region/World"]
    )
    .transform_window(
        rank="rank(AvgPrevalence)",
        sort=[alt.SortField("AvgPrevalence", order="descending")]
    )
    .transform_filter(alt.datum.rank <= 12)
    .mark_bar()
    .encode(
        x=alt.X("AvgPrevalence:Q", title="Average Prevalence (%)"),
        y=alt.Y("Country/Region/World:N", sort="-x", title="Country"),
        tooltip=[
            alt.Tooltip("Country/Region/World:N", title="Country"),
            alt.Tooltip("AvgPrevalence:Q", title="Average Prevalence (%)", format=".1f")
        ]
    )
    .properties(
        width=360,
        height=420,
        title="Top selected countries"
    )
)

chart4b = (scatter2 | bars2).properties(
    title="Linked View: Economic Development and Cardiometabolic Risk"
).configure_title(
    fontSize=18,
    anchor="start"
).configure_axis(
    labelFontSize=11,
    titleFontSize=13
).configure_legend(
    titleFontSize=12,
    labelFontSize=11
).configure_view(
    stroke=None
)

chart4b

### Visual 5
Linked View: Ageing Population vs Raised Blood Pressure

In [ ]:
age_data_5 = master.dropna(
    subset=["Age65PlusPercent", "PrevalencePercent", "Region"]
).copy()

age_data_5 = age_data_5[
    (age_data_5["Indicator"] == "Raised Blood Pressure") &
    (age_data_5["Region"] != "Other")
]

region_options_5 = sorted(age_data_5["Region"].dropna().unique())

region_dropdown_5 = alt.param(
    name="RegionFilter5",
    bind=alt.binding_select(options=["All"] + region_options_5, name="Region: "),
    value="All"
)

year_slider_5 = alt.param(
    name="YearFilter5",
    bind=alt.binding_range(
        min=int(age_data_5["Year"].min()),
        max=int(age_data_5["Year"].max()),
        step=1,
        name="Year: "
    ),
    value=2010
)

brush_5 = alt.selection_interval(name="BrushSelect5")

base_5 = (
    alt.Chart(age_data_5)
    .add_params(region_dropdown_5, year_slider_5, brush_5)
    .transform_filter(alt.datum.Year == year_slider_5)
    .transform_filter(
        "(RegionFilter5 == 'All') || (datum.Region == RegionFilter5)"
    )
)

scatter_5 = base_5.mark_circle(size=95).encode(
    x=alt.X(
        "Age65PlusPercent:Q",
        title="Population Aged 65 and Above (%)"
    ),
    y=alt.Y(
        "PrevalencePercent:Q",
        title="Raised Blood Pressure Prevalence (%)"
    ),
    color=alt.condition(
        brush_5,
        alt.Color("Region:N", title="Region"),
        alt.value("lightgray")
    ),
    tooltip=[
        alt.Tooltip("Country/Region/World:N", title="Country"),
        alt.Tooltip("Region:N", title="Region"),
        alt.Tooltip("Year:Q", title="Year"),
        alt.Tooltip("Age65PlusPercent:Q", title="Age 65+ (%)", format=".1f"),
        alt.Tooltip("PrevalencePercent:Q", title="Raised Blood Pressure (%)", format=".1f")
    ]
).properties(
    width=520,
    height=420,
)

bars_5 = (
    base_5.transform_filter(brush_5)
    .transform_window(
        rank="rank(PrevalencePercent)",
        sort=[alt.SortField("PrevalencePercent", order="descending")]
    )
    .transform_filter(alt.datum.rank <= 12)
    .mark_bar()
    .encode(
        x=alt.X("PrevalencePercent:Q", title="Raised Blood Pressure (%)"),
        y=alt.Y("Country/Region/World:N", sort="-x", title="Country"),
        tooltip=[
            alt.Tooltip("Country/Region/World:N", title="Country"),
            alt.Tooltip("PrevalencePercent:Q", title="Raised Blood Pressure (%)", format=".1f")
        ]
    )
    .properties(
        width=360,
        height=420,
        title="Top selected countries"
    )
)

chart_5 = (scatter_5 | bars_5).properties(
    title="Linked View: Ageing Population and Raised Blood Pressure"
).configure_title(
    fontSize=18,
    anchor="start"
).configure_axis(
    labelFontSize=11,
    titleFontSize=13
).configure_legend(
    titleFontSize=12,
    labelFontSize=11
)

chart_5

### Visual 6
Linked View: Urbanisation and Obesity

In [ ]:
import altair as alt
import pandas as pd

viz06_data = master.dropna(
    subset=["UrbanPopulationPercent", "PrevalencePercent", "Region"]
).copy()

viz06_data = viz06_data[
    (viz06_data["Indicator"] == "Obesity") &
    (viz06_data["Region"] != "Other")
]

# Aggregate across sex so each country-year has one row
viz06_data_agg = (
    viz06_data.groupby(
        ["Country/Region/World", "Region", "Year"],
        as_index=False
    )
    .agg(
        UrbanPopulationPercent=("UrbanPopulationPercent", "mean"),
        PrevalencePercent=("PrevalencePercent", "mean")
    )
)

viz06_region_options = sorted(viz06_data_agg["Region"].dropna().unique())

viz06_param_region = alt.param(
    name="Viz06RegionFilter",
    bind=alt.binding_select(
        options=["All"] + viz06_region_options,
        name="Region: "
    ),
    value="All"
)

viz06_param_year = alt.param(
    name="Viz06YearFilter",
    bind=alt.binding_range(
        min=int(viz06_data_agg["Year"].min()),
        max=int(viz06_data_agg["Year"].max()),
        step=1,
        name="Year: "
    ),
    value=2010
)

viz06_brush = alt.selection_interval(name="Viz06Brush")

viz06_base = (
    alt.Chart(viz06_data_agg)
    .add_params(viz06_param_region, viz06_param_year, viz06_brush)
    .transform_filter(alt.datum.Year == viz06_param_year)
    .transform_filter(
        "(Viz06RegionFilter == 'All') || (datum.Region == Viz06RegionFilter)"
    )
)

viz06_scatter = viz06_base.mark_circle(size=95).encode(
    x=alt.X(
        "UrbanPopulationPercent:Q",
        title="Urban Population (%)"
    ),
    y=alt.Y(
        "PrevalencePercent:Q",
        title="Obesity Prevalence (%)"
    ),
    color=alt.condition(
        viz06_brush,
        alt.Color("Region:N", title="Region"),
        alt.value("lightgray")
    ),
    tooltip=[
        alt.Tooltip("Country/Region/World:N", title="Country"),
        alt.Tooltip("Region:N", title="Region"),
        alt.Tooltip("Year:Q", title="Year"),
        alt.Tooltip("UrbanPopulationPercent:Q", title="Urban Population (%)", format=".1f"),
        alt.Tooltip("PrevalencePercent:Q", title="Obesity Prevalence (%)", format=".1f")
    ]
).properties(
    width=520,
    height=420
)

viz06_bars = (
    viz06_base.transform_filter(viz06_brush)
    .transform_window(
        rank="rank(PrevalencePercent)",
        sort=[alt.SortField("PrevalencePercent", order="descending")]
    )
    .transform_filter(alt.datum.rank <= 12)
    .mark_bar()
    .encode(
        x=alt.X("PrevalencePercent:Q", title="Obesity Prevalence (%)"),
        y=alt.Y("Country/Region/World:N", sort="-x", title="Country"),
        tooltip=[
            alt.Tooltip("Country/Region/World:N", title="Country"),
            alt.Tooltip("PrevalencePercent:Q", title="Obesity Prevalence (%)", format=".1f")
        ]
    )
    .properties(
        width=360,
        height=420,
        title="Top selected countries"
    )
)

viz06 = (viz06_scatter | viz06_bars).properties(
    title="Linked View: Urbanisation and Obesity"
).configure_title(
    fontSize=18,
    anchor="start"
).configure_axis(
    labelFontSize=11,
    titleFontSize=13
).configure_legend(
    titleFontSize=12,
    labelFontSize=11
).configure_view(
    stroke=None
)

viz06

### Visual 7
Income Group Comparison by Indicator and Sex

In [ ]:
viz07_data = master.dropna(subset=["IncomeGroup", "PrevalencePercent"]).copy()

viz07_income_order = [
    "Low income",
    "Lower middle income",
    "Upper middle income",
    "High income"
]

viz07 = (
    alt.Chart(viz07_data)
    .mark_boxplot(size=16, outliers=False)
    .encode(
        x=alt.X(
            "IncomeGroup:N",
            title="Income Group",
            sort=viz07_income_order,
            scale=alt.Scale(paddingInner=0.35, paddingOuter=0.2)
        ),
        xOffset=alt.XOffset(
            "Sex:N",
            scale=alt.Scale(paddingInner=0.5)
        ),
        y=alt.Y(
            "PrevalencePercent:Q",
            title="Prevalence (%)"
        ),
        color=alt.Color(
            "Sex:N",
            title="Sex",
            scale=alt.Scale(
                domain=["Men", "Women"],
                range=["#4C78A8", "#E45756"]
            )
        ),
        column=alt.Column(
            "Indicator:N",
            title="Indicator"
        ),
        tooltip=[
            alt.Tooltip("IncomeGroup:N", title="Income Group"),
            alt.Tooltip("Sex:N", title="Sex"),
            alt.Tooltip("Indicator:N", title="Indicator"),
            alt.Tooltip("PrevalencePercent:Q", title="Prevalence (%)", format=".1f")
        ]
    )
    .properties(
        width=250,
        height=400,
        title="Distribution of Cardiometabolic Risk by Income Group and Sex"
    )
    .configure_title(
        fontSize=18,
        anchor="start"
    )
    .configure_axis(
        labelFontSize=11,
        titleFontSize=13
    )
    .configure_legend(
        titleFontSize=12,
        labelFontSize=11
    )
    .configure_view(
        stroke=None
    )
)

viz07

### Visual 8
Regional Ranking of Obesity Over Time

In [ ]:
viz08_data = master.dropna(subset=["Region","PrevalencePercent"]).copy()

viz08_data = viz08_data[viz08_data["Region"] != "Other"]

viz08_region_avg = (
    viz08_data
    .groupby(["Year","Region","Indicator"], as_index=False)["PrevalencePercent"]
    .mean()
)

viz08_indicator_options = sorted(viz08_region_avg["Indicator"].unique())

viz08_indicator_dropdown = alt.param(
    name="SelectIndicatorViz08",
    bind=alt.binding_select(
        options=viz08_indicator_options,
        name="Indicator: "
    ),
    value=viz08_indicator_options[0]
)

viz08 = (
    alt.Chart(viz08_region_avg)
    .add_params(viz08_indicator_dropdown)
    .transform_filter(alt.datum.Indicator == viz08_indicator_dropdown)
    .transform_window(
        rank="rank(PrevalencePercent)",
        sort=[alt.SortField("PrevalencePercent", order="descending")],
        groupby=["Year"]
    )
    .mark_line(point=True)
    .encode(
        x=alt.X("Year:O", title="Year"),
        y=alt.Y(
            "rank:Q",
            title="Regional Rank",
            scale=alt.Scale(reverse=True)
        ),
        color=alt.Color("Region:N", title="Region"),
        tooltip=[
            alt.Tooltip("Year:O", title="Year"),
            alt.Tooltip("Region:N", title="Region"),
            alt.Tooltip("rank:Q", title="Rank"),
            alt.Tooltip("PrevalencePercent:Q", title="Average Prevalence (%)", format=".1f")
        ]
    )
    .properties(
        width=850,
        height=420,
        title="Regional Ranking of Cardiometabolic Risk Over Time"
    )
    .configure_title(
        fontSize=18,
        anchor="start"
    )
)

viz08

### Visual 9
Bubble Chart: GDP vs Health Risk with Ageing as Bubble Size

In [ ]:
viz09_data = master.dropna(
    subset=["GDPPerCapita", "PrevalencePercent", "Age65PlusPercent", "Region"]
).copy()

viz09_data = viz09_data[viz09_data["Region"] != "Other"]

# Aggregate across sex so each country-year-indicator has one row
viz09_data_agg = (
    viz09_data.groupby(
        ["Country/Region/World", "Region", "Year", "Indicator"],
        as_index=False
    )
    .agg(
        GDPPerCapita=("GDPPerCapita", "mean"),
        PrevalencePercent=("PrevalencePercent", "mean"),
        Age65PlusPercent=("Age65PlusPercent", "mean")
    )
)

viz09_indicator_options = sorted(viz09_data_agg["Indicator"].unique())

viz09_param_indicator = alt.param(
    name="Viz09IndicatorFilter",
    bind=alt.binding_select(
        options=viz09_indicator_options,
        name="Indicator: "
    ),
    value=viz09_indicator_options[0]
)

viz09_param_year = alt.param(
    name="Viz09YearFilter",
    bind=alt.binding_range(
        min=int(viz09_data_agg["Year"].min()),
        max=int(viz09_data_agg["Year"].max()),
        step=1,
        name="Year: "
    ),
    value=2010
)

viz09 = (
    alt.Chart(viz09_data_agg)
    .add_params(viz09_param_indicator, viz09_param_year)
    .transform_filter(alt.datum.Indicator == viz09_param_indicator)
    .transform_filter(alt.datum.Year == viz09_param_year)
    .mark_circle(opacity=0.7)
    .encode(
        x=alt.X(
            "GDPPerCapita:Q",
            title="GDP per Capita",
            scale=alt.Scale(type="log")
        ),
        y=alt.Y(
            "PrevalencePercent:Q",
            title="Prevalence (%)"
        ),
        size=alt.Size(
            "Age65PlusPercent:Q",
            title="Population Aged 65+ (%)",
            scale=alt.Scale(range=[30, 1200])
        ),
        color=alt.Color(
            "Region:N",
            title="Region"
        ),
        tooltip=[
            alt.Tooltip("Country/Region/World:N", title="Country"),
            alt.Tooltip("Region:N", title="Region"),
            alt.Tooltip("Indicator:N", title="Indicator"),
            alt.Tooltip("Year:Q", title="Year"),
            alt.Tooltip("GDPPerCapita:Q", title="GDP per Capita", format=",.0f"),
            alt.Tooltip("PrevalencePercent:Q", title="Prevalence (%)", format=".1f"),
            alt.Tooltip("Age65PlusPercent:Q", title="Population Aged 65+ (%)", format=".1f")
        ]
    )
    .properties(
        width=780,
        height=400,
        title="Economic Development, Ageing, and Cardiometabolic Risk"
    )
    .configure_title(
        fontSize=18,
        anchor="start"
    )
    .configure_axis(
        labelFontSize=11,
        titleFontSize=13
    )
    .configure_legend(
        titleFontSize=12,
        labelFontSize=11
    )
    .configure_view(
        stroke=None
    )
)

viz09

### Visual 10
Correlation heatmap between cardiometabolic risk and socioeconomic factors

In [ ]:
corr_data = master[[
    "PrevalencePercent",
    "GDPPerCapita",
    "HealthExpenditurePerCapita",
    "UrbanPopulationPercent",
    "Age65PlusPercent"
]].dropna()

corr_matrix = corr_data.corr().reset_index().melt(id_vars="index")

corr_matrix.columns = ["Variable1", "Variable2", "Correlation"]

In [ ]:
import pandas as pd
import altair as alt

corr_base = master[master["Country/Region/World"] != "World"].copy()

corr_grouped = (
    corr_base.groupby(["Indicator", "Country/Region/World", "Year"], as_index=False)
    .agg({
        "PrevalencePercent": "mean",
        "GDPPerCapita": "mean",
        "HealthExpenditurePerCapita": "mean",
        "UrbanPopulationPercent": "mean",
        "Age65PlusPercent": "mean"
    })
)

indicator_options = sorted(corr_grouped["Indicator"].dropna().unique().tolist())

corr_param = alt.param(
    name="IndicatorSelect",
    bind=alt.binding_select(options=indicator_options, name="Indicator: "),
    value=indicator_options[0]
)

corr_charts = []

variables = [
    "PrevalencePercent",
    "GDPPerCapita",
    "HealthExpenditurePerCapita",
    "UrbanPopulationPercent",
    "Age65PlusPercent"
]

for indicator in indicator_options:
    df_sub = corr_grouped[corr_grouped["Indicator"] == indicator][variables]
    corr = df_sub.corr()
    melted = corr.reset_index().melt(id_vars="index")
    melted.columns = ["Variable1", "Variable2", "Correlation"]
    melted["Indicator"] = indicator
    corr_charts.append(melted)

corr_matrix_all = pd.concat(corr_charts, ignore_index=True)

heatmap = (
    alt.Chart(corr_matrix_all)
    .add_params(corr_param)
    .transform_filter(alt.datum.Indicator == corr_param)
    .mark_rect()
    .encode(
        x=alt.X("Variable1:N", title=""),
        y=alt.Y("Variable2:N", title=""),
        color=alt.Color(
            "Correlation:Q",
            scale=alt.Scale(scheme="redblue", domain=[-1, 1]),
            title="Correlation"
        ),
        tooltip=[
            alt.Tooltip("Indicator:N", title="Indicator"),
            alt.Tooltip("Variable1:N", title="Variable 1"),
            alt.Tooltip("Variable2:N", title="Variable 2"),
            alt.Tooltip("Correlation:Q", format=".2f")
        ]
    )
    .properties(
        width=420,
        height=420,
        title="Correlation Between Selected Health Indicator and Socioeconomic Factors"
    )
)

text = (
    alt.Chart(corr_matrix_all)
    .add_params(corr_param)
    .transform_filter(alt.datum.Indicator == corr_param)
    .mark_text(size=12)
    .encode(
        x="Variable1:N",
        y="Variable2:N",
        text=alt.Text("Correlation:Q", format=".2f"),
        color=alt.condition(
            "abs(datum.Correlation) > 0.5",
            alt.value("white"),
            alt.value("black")
        )
    )
)

viz10 = (heatmap + text).configure_axis(labelAngle=-40)

viz10

### Visual 11
Top 15 Countries by Prevalence

In [ ]:
viz11_data = master.dropna(subset=["PrevalencePercent"]).copy()
viz11_data = viz11_data[viz11_data["Country/Region/World"] != "World"]

# Aggregate across sex so each country-year-indicator has one row
viz11_data_agg = (
    viz11_data.groupby(
        ["Country/Region/World", "Region", "Year", "Indicator"],
        as_index=False
    )
    .agg(
        PrevalencePercent=("PrevalencePercent", "mean")
    )
)

viz11_indicator_options = sorted(viz11_data_agg["Indicator"].unique())

viz11_param_indicator = alt.param(
    name="Viz11IndicatorFilter",
    bind=alt.binding_select(
        options=viz11_indicator_options,
        name="Indicator: "
    ),
    value=viz11_indicator_options[0]
)

viz11_param_year = alt.param(
    name="Viz11YearFilter",
    bind=alt.binding_range(
        min=int(viz11_data_agg["Year"].min()),
        max=int(viz11_data_agg["Year"].max()),
        step=1,
        name="Year: "
    ),
    value=2010
)

viz11 = (
    alt.Chart(viz11_data_agg)
    .add_params(viz11_param_indicator, viz11_param_year)
    .transform_filter(alt.datum.Indicator == viz11_param_indicator)
    .transform_filter(alt.datum.Year == viz11_param_year)
    .transform_window(
        rank="rank(PrevalencePercent)",
        sort=[alt.SortField("PrevalencePercent", order="descending")]
    )
    .transform_filter(alt.datum.rank <= 15)
    .mark_bar()
    .encode(
        x=alt.X("PrevalencePercent:Q", title="Prevalence (%)"),
        y=alt.Y(
            "Country/Region/World:N",
            sort="-x",
            title="Country"
        ),
        color=alt.Color(
            "Region:N",
            title="Region"
        ),
        tooltip=[
            alt.Tooltip("Country/Region/World:N", title="Country"),
            alt.Tooltip("Region:N", title="Region"),
            alt.Tooltip("Indicator:N", title="Indicator"),
            alt.Tooltip("Year:Q", title="Year"),
            alt.Tooltip("PrevalencePercent:Q", title="Prevalence (%)", format=".1f")
        ]
    )
    .properties(
        width=780,
        height=460,
        title="Top 15 Countries by Cardiometabolic Risk Prevalence"
    )
    .configure_title(
        fontSize=18,
        anchor="start"
    )
    .configure_axis(
        labelFontSize=11,
        titleFontSize=13
    )
    .configure_legend(
        titleFontSize=12,
        labelFontSize=11
    )
    .configure_view(
        stroke=None
    )
)

viz11

### Visual 12
Interactive Dumbbell Chart: Gender gap in prevalence across countries

In [ ]:
# =========================
# Visual 12
# Interactive Dumbbell Chart
# Gender gap in prevalence across countries
# =========================

import altair as alt
import pandas as pd
import numpy as np

# -------------------------
# Prepare data
# -------------------------
v12_df = master.copy()

v12_df = v12_df[
    [
        "Country/Region/World",
        "Sex",
        "Year",
        "Indicator",
        "PrevalencePercent",
        "Region"
    ]
].dropna()

v12_df = (
    v12_df
    .groupby(
        ["Country/Region/World", "Sex", "Year", "Indicator", "Region"],
        as_index=False
    )["PrevalencePercent"]
    .mean()
)

indicator_options = sorted(v12_df["Indicator"].unique().tolist())
year_options = sorted(v12_df["Year"].unique().tolist())

default_indicator = "Obesity" if "Obesity" in indicator_options else indicator_options[0]
default_year = 2016 if 2016 in year_options else year_options[-1]

sex_values = sorted(v12_df["Sex"].unique().tolist())

if "Men" in sex_values and "Women" in sex_values:
    male_label = "Men"
    female_label = "Women"
elif "Male" in sex_values and "Female" in sex_values:
    male_label = "Male"
    female_label = "Female"
else:
    raise ValueError(f"Unexpected Sex labels found: {sex_values}")

# -------------------------
# Interactive controls
# -------------------------
indicator_param = alt.selection_point(
    fields=["Indicator"],
    bind=alt.binding_select(
        options=indicator_options,
        name="Indicator: "
    ),
    value=[{"Indicator": default_indicator}]
)

year_min = int(v12_df["Year"].min())
year_max = int(v12_df["Year"].max())

year_param = alt.selection_point(
    fields=["Year"],
    bind=alt.binding_range(
        min=year_min,
        max=year_max,
        step=1,
        name="Year: "
    ),
    value=[{"Year": default_year}]
)

# -------------------------
# Base chart with filters
# -------------------------
base = (
    alt.Chart(v12_df)
    .transform_filter(indicator_param)
    .transform_filter(year_param)
)

# -------------------------
# Dumbbell data
# -------------------------
gap_base = (
    base
    .transform_pivot(
        pivot="Sex",
        value="PrevalencePercent",
        groupby=["Country/Region/World", "Year", "Indicator", "Region"]
    )
    .transform_calculate(
        male=f"datum['{male_label}']",
        female=f"datum['{female_label}']",
        gap=f"datum['{female_label}'] - datum['{male_label}']",
        abs_gap=f"abs(datum['{female_label}'] - datum['{male_label}'])"
    )
    .transform_filter("isValid(datum.male) && isValid(datum.female)")
    .transform_window(
        rank="rank()",
        sort=[alt.SortField("abs_gap", order="descending")]
    )
    .transform_filter("datum.rank <= 20")
)

# -------------------------
# Global averages for reference lines
# -------------------------
global_avg = (
    base
    .transform_aggregate(
        avg_prev="mean(PrevalencePercent)",
        groupby=["Sex"]
    )
)

male_avg_rule = (
    global_avg
    .transform_filter(f"datum.Sex == '{male_label}'")
    .mark_rule(
        strokeDash=[6, 4],
        strokeWidth=1.5,
        color="#4C78A8",
        opacity=0.7
    )
    .encode(
        x="avg_prev:Q"
    )
)

female_avg_rule = (
    global_avg
    .transform_filter(f"datum.Sex == '{female_label}'")
    .mark_rule(
        strokeDash=[6, 4],
        strokeWidth=1.5,
        color="#E45756",
        opacity=0.7
    )
    .encode(
        x="avg_prev:Q"
    )
)

# -------------------------
# Dumbbell layers
# -------------------------
rules = gap_base.mark_rule(
    strokeWidth=3,
    opacity=0.4
).encode(
    y=alt.Y(
        "Country/Region/World:N",
        sort=alt.SortField(field="abs_gap", order="descending"),
        title=None
    ),
    x=alt.X(
        "male:Q",
        title="Prevalence (%)",
        scale=alt.Scale(zero=False)
    ),
    x2="female:Q",
    tooltip=[
        alt.Tooltip("Country/Region/World:N", title="Country"),
        alt.Tooltip("Region:N", title="Region"),
        alt.Tooltip("male:Q", title=f"{male_label} (%)", format=".2f"),
        alt.Tooltip("female:Q", title=f"{female_label} (%)", format=".2f"),
        alt.Tooltip("gap:Q", title=f"{female_label} minus {male_label}", format=".2f")
    ]
)

male_points = (
    gap_base
    .transform_calculate(SexLegend=f"'{male_label}'")
    .mark_circle(
        size=120,
        opacity=0.95
    )
    .encode(
        y=alt.Y(
            "Country/Region/World:N",
            sort=alt.SortField(field="abs_gap", order="descending"),
            title=None
        ),
        x="male:Q",
        color=alt.Color(
            "SexLegend:N",
            scale=alt.Scale(
                domain=[male_label, female_label],
                range=["#4C78A8", "#E45756"]
            ),
            legend=alt.Legend(
                title="Sex",
                orient="right"
            )
        ),
        tooltip=[
            alt.Tooltip("Country/Region/World:N", title="Country"),
            alt.Tooltip("Region:N", title="Region"),
            alt.Tooltip("male:Q", title=f"{male_label} (%)", format=".2f"),
            alt.Tooltip("female:Q", title=f"{female_label} (%)", format=".2f"),
            alt.Tooltip("gap:Q", title=f"{female_label} minus {male_label}", format=".2f")
        ]
    )
)

female_points = (
    gap_base
    .transform_calculate(SexLegend=f"'{female_label}'")
    .mark_circle(
        size=120,
        opacity=0.95
    )
    .encode(
        y=alt.Y(
            "Country/Region/World:N",
            sort=alt.SortField(field="abs_gap", order="descending"),
            title=None
        ),
        x="female:Q",
        color=alt.Color(
            "SexLegend:N",
            scale=alt.Scale(
                domain=[male_label, female_label],
                range=["#4C78A8", "#E45756"]
            ),
            legend=None
        ),
        tooltip=[
            alt.Tooltip("Country/Region/World:N", title="Country"),
            alt.Tooltip("Region:N", title="Region"),
            alt.Tooltip("male:Q", title=f"{male_label} (%)", format=".2f"),
            alt.Tooltip("female:Q", title=f"{female_label} (%)", format=".2f"),
            alt.Tooltip("gap:Q", title=f"{female_label} minus {male_label}", format=".2f")
        ]
    )
)

gap_labels = (
    gap_base
    .transform_filter("datum.rank <= 8")
    .mark_text(
        align="left",
        dx=8,
        fontSize=11,
        fontWeight="bold",
        color="#333333"
    )
    .encode(
        y=alt.Y(
            "Country/Region/World:N",
            sort=alt.SortField(field="abs_gap", order="descending"),
            title=None
        ),
        x="female:Q",
        text=alt.Text("gap:Q", format=".2f")
    )
)

# -------------------------
# Final chart
# -------------------------
visual12 = (
    (rules + male_points + female_points + gap_labels + male_avg_rule + female_avg_rule)
    .add_params(indicator_param, year_param)
    .properties(
        width=850,
        height=520,
        title={
            "text": "Gender gap in prevalence across countries",
            "subtitle": [
                "Top 20 countries ranked by absolute difference between male and female prevalence",
                "Dashed vertical lines show global averages for each sex"
            ]
        }
    )
    .configure_title(
        anchor="start",
        fontSize=18,
        subtitleFontSize=12
    )
    .configure_axis(
        labelFontSize=11,
        titleFontSize=12
    )
    .configure_view(
        stroke=None
    )
)

visual12

### Visual 13
Linked Heatmap + Trend Chart: Prevalence patterns across income groups over time

In [ ]:
# =========================
# Visual 13
# Linked Heatmap + Trend Chart
# Prevalence patterns across income groups over time
# =========================

import altair as alt
import pandas as pd
import numpy as np

# -------------------------
# Prepare data
# -------------------------
v13_df = master.copy()

v13_df = v13_df[
    [
        "IncomeGroup",
        "Year",
        "Indicator",
        "PrevalencePercent"
    ]
].dropna()

# Remove blanks if any
v13_df = v13_df[v13_df["IncomeGroup"].astype(str).str.strip() != ""]

# Aggregate mean prevalence by income group, year, indicator
v13_df = (
    v13_df
    .groupby(["IncomeGroup", "Year", "Indicator"], as_index=False)["PrevalencePercent"]
    .mean()
)

indicator_options = sorted(v13_df["Indicator"].unique().tolist())
default_indicator = "Obesity" if "Obesity" in indicator_options else indicator_options[0]

# Fixed order if available
preferred_income_order = [
    "Low income",
    "Lower middle income",
    "Upper middle income",
    "High income"
]

actual_income_groups = v13_df["IncomeGroup"].unique().tolist()
income_order = [x for x in preferred_income_order if x in actual_income_groups]

# Add any extra categories not in preferred list
remaining_groups = sorted([x for x in actual_income_groups if x not in income_order])
income_order = income_order + remaining_groups

# -------------------------
# Interactive controls
# -------------------------
indicator_param = alt.selection_point(
    fields=["Indicator"],
    bind=alt.binding_select(
        options=indicator_options,
        name="Indicator: "
    ),
    value=[{"Indicator": default_indicator}]
)

income_click = alt.selection_point(
    fields=["IncomeGroup"],
    empty=True
)

# -------------------------
# Base chart
# -------------------------
base = (
    alt.Chart(v13_df)
    .transform_filter(indicator_param)
)

# -------------------------
# Heatmap
# -------------------------
heatmap = base.mark_rect(cornerRadius=2).encode(
    x=alt.X(
        "Year:O",
        title="Year",
        axis=alt.Axis(labelAngle=-45)
    ),
    y=alt.Y(
        "IncomeGroup:N",
        title=None,
        sort=income_order
    ),
    color=alt.Color(
        "PrevalencePercent:Q",
        title="Avg prevalence (%)"
    ),
    opacity=alt.condition(income_click, alt.value(1), alt.value(0.85)),
    tooltip=[
        alt.Tooltip("IncomeGroup:N", title="Income group"),
        alt.Tooltip("Year:O", title="Year"),
        alt.Tooltip("PrevalencePercent:Q", title="Average prevalence (%)", format=".0f")
    ]
).properties(
    width=850,
    height=180
)

heatmap_text = base.mark_text(fontSize=10).encode(
    x=alt.X("Year:O"),
    y=alt.Y("IncomeGroup:N", sort=income_order),
    text=alt.Text("PrevalencePercent:Q", format=".0f"),
    color=alt.condition(
        alt.datum.PrevalencePercent > v13_df["PrevalencePercent"].median(),
        alt.value("black"),
        alt.value("black")
    )
)

heatmap_layer = (
    (heatmap + heatmap_text)
    .add_params(income_click)
    .encode(
        stroke=alt.condition(income_click, alt.value("black"), alt.value(None))
    )
)

# -------------------------
# Trend chart
# -------------------------
trend_lines = base.mark_line(point=False, strokeWidth=2).encode(
    x=alt.X("Year:O", title="Year"),
    y=alt.Y("PrevalencePercent:Q", title="Average prevalence (%)"),
    color=alt.condition(
        income_click,
        alt.Color(
            "IncomeGroup:N",
            sort=income_order,
            legend=alt.Legend(title="Income group")
        ),
        alt.value("#C7C7C7")
    ),
    detail="IncomeGroup:N",
    opacity=alt.condition(income_click, alt.value(1), alt.value(0.35)),
    tooltip=[
        alt.Tooltip("IncomeGroup:N", title="Income group"),
        alt.Tooltip("Year:O", title="Year"),
        alt.Tooltip("PrevalencePercent:Q", title="Average prevalence (%)", format=".2f")
    ]
).properties(
    width=850,
    height=280
)

highlight_points = base.mark_circle(size=85).encode(
    x=alt.X("Year:O"),
    y=alt.Y("PrevalencePercent:Q"),
    color=alt.Color(
        "IncomeGroup:N",
        sort=income_order,
        legend=None
    ),
    opacity=alt.condition(income_click, alt.value(1), alt.value(0))
).transform_filter(
    income_click
)

# -------------------------
# Optional helper rule on hover
# -------------------------
hover = alt.selection_point(
    nearest=True,
    on="mouseover",
    fields=["Year"],
    empty=False
)

hover_rule = alt.Chart(v13_df).mark_rule(color="gray", opacity=0.25).encode(
    x=alt.X("Year:O")
).transform_filter(
    indicator_param
).add_params(
    hover
).transform_filter(
    hover
)

# -------------------------
# Combine
# -------------------------
visual13 = (
    alt.vconcat(
        (heatmap + heatmap_text)
        .add_params(indicator_param, income_click)
        .properties(
            title={
                "text": "Prevalence patterns across income groups over time",
                "subtitle": [
                    "Heatmap overview with linked trend chart below",
                    "Click an income group in the heatmap to highlight its trend"
                ]
            }
        ),
        (trend_lines + highlight_points)
    ,
        spacing=20
    )
    .resolve_scale(color="independent")
    .configure_title(
        anchor="start",
        fontSize=18,
        subtitleFontSize=12
    )
    .configure_axis(
        labelFontSize=11,
        titleFontSize=12
    )
    .configure_view(
        stroke=None
    )
)

visual13

### Visual 14
Ranked Lollipop Change Chart: Countries with the fastest rise in prevalence

In [ ]:
# =========================
# Visual 14
# Ranked Lollipop Change Chart
# Countries with the fastest rise in prevalence
# =========================

import altair as alt
import pandas as pd
import numpy as np

# -------------------------
# Prepare data
# -------------------------
v14_df = master.copy()

v14_df = v14_df[
    [
        "Country/Region/World",
        "Year",
        "Indicator",
        "PrevalencePercent",
        "Region"
    ]
].dropna()

# Exclude obvious aggregate rows if present
exclude_values = [
    "World", "Africa", "Asia", "Europe", "North America",
    "South America", "Oceania"
]
v14_df = v14_df[~v14_df["Country/Region/World"].isin(exclude_values)].copy()

# Aggregate safely
v14_df = (
    v14_df
    .groupby(
        ["Country/Region/World", "Year", "Indicator", "Region"],
        as_index=False
    )["PrevalencePercent"]
    .mean()
)

indicator_options = sorted(v14_df["Indicator"].unique().tolist())
year_options = sorted(v14_df["Year"].unique().tolist())

default_indicator = "Obesity" if "Obesity" in indicator_options else indicator_options[0]
default_start_year = 1975 if 1975 in year_options else min(year_options)
default_end_year = 2016 if 2016 in year_options else max(year_options)

# -------------------------
# Build year-pair comparison table in pandas
# -------------------------
records = []

for indicator in indicator_options:
    temp = v14_df[v14_df["Indicator"] == indicator].copy()

    wide = (
        temp
        .pivot_table(
            index=["Country/Region/World", "Region"],
            columns="Year",
            values="PrevalencePercent",
            aggfunc="mean"
        )
        .reset_index()
    )

    available_years = [y for y in year_options if y in wide.columns]

    for sy in available_years:
        for ey in available_years:
            if sy < ey:
                tmp = wide[["Country/Region/World", "Region", sy, ey]].dropna().copy()
                tmp["Indicator"] = indicator
                tmp["StartYear"] = sy
                tmp["EndYear"] = ey
                tmp["start_prev"] = tmp[sy]
                tmp["end_prev"] = tmp[ey]
                tmp["change"] = tmp["end_prev"] - tmp["start_prev"]
                tmp = tmp.drop(columns=[sy, ey])

                # Keep only increases for this visual
                tmp = tmp[tmp["change"] > 0]

                records.append(tmp)

v14_change_df = pd.concat(records, ignore_index=True)

# -------------------------
# Explicitly named parameters
# -------------------------
v14_indicator_param = alt.selection_point(
    name="v14_indicator_param",
    fields=["Indicator"],
    bind=alt.binding_select(
        options=indicator_options,
        name="Indicator: "
    ),
    value=[{"Indicator": default_indicator}]
)

v14_start_param = alt.selection_point(
    name="v14_start_param",
    fields=["StartYear"],
    bind=alt.binding_select(
        options=year_options,
        name="Start year: "
    ),
    value=[{"StartYear": default_start_year}]
)

v14_end_param = alt.selection_point(
    name="v14_end_param",
    fields=["EndYear"],
    bind=alt.binding_select(
        options=year_options,
        name="End year: "
    ),
    value=[{"EndYear": default_end_year}]
)

# -------------------------
# Base chart
# -------------------------
v14_base = (
    alt.Chart(v14_change_df)
    .transform_filter(v14_indicator_param)
    .transform_filter(v14_start_param)
    .transform_filter(v14_end_param)
    .transform_window(
        rank="rank()",
        sort=[alt.SortField("change", order="descending")]
    )
    .transform_filter("datum.rank <= 15")
)

# -------------------------
# Lollipop stems
# -------------------------
stems = v14_base.mark_rule(
    strokeWidth=3,
    opacity=0.4
).encode(
    y=alt.Y(
        "Country/Region/World:N",
        sort=alt.SortField(field="change", order="descending"),
        title=None
    ),
    x=alt.X(
        "start_prev:Q",
        title="Prevalence (%)",
        scale=alt.Scale(zero=False)
    ),
    x2="end_prev:Q",
    tooltip=[
        alt.Tooltip("Country/Region/World:N", title="Country"),
        alt.Tooltip("Region:N", title="Region"),
        alt.Tooltip("StartYear:O", title="Start year"),
        alt.Tooltip("start_prev:Q", title="Start prevalence (%)", format=".2f"),
        alt.Tooltip("EndYear:O", title="End year"),
        alt.Tooltip("end_prev:Q", title="End prevalence (%)", format=".2f"),
        alt.Tooltip("change:Q", title="Increase", format=".2f")
    ]
)

# Start points
start_points = v14_base.mark_circle(
    size=90,
    opacity=0.9
).encode(
    y=alt.Y(
        "Country/Region/World:N",
        sort=alt.SortField(field="change", order="descending"),
        title=None
    ),
    x="start_prev:Q",
    color=alt.value("#9E9E9E"),
    tooltip=[
        alt.Tooltip("Country/Region/World:N", title="Country"),
        alt.Tooltip("start_prev:Q", title="Start prevalence (%)", format=".2f"),
        alt.Tooltip("end_prev:Q", title="End prevalence (%)", format=".2f"),
        alt.Tooltip("change:Q", title="Increase", format=".2f")
    ]
)

# End points
end_points = v14_base.mark_circle(
    size=130,
    opacity=0.95
).encode(
    y=alt.Y(
        "Country/Region/World:N",
        sort=alt.SortField(field="change", order="descending"),
        title=None
    ),
    x="end_prev:Q",
    color=alt.Color(
        "Region:N",
        legend=alt.Legend(title="Region")
    ),
    tooltip=[
        alt.Tooltip("Country/Region/World:N", title="Country"),
        alt.Tooltip("Region:N", title="Region"),
        alt.Tooltip("StartYear:O", title="Start year"),
        alt.Tooltip("start_prev:Q", title="Start prevalence (%)", format=".2f"),
        alt.Tooltip("EndYear:O", title="End year"),
        alt.Tooltip("end_prev:Q", title="End prevalence (%)", format=".2f"),
        alt.Tooltip("change:Q", title="Increase", format=".2f")
    ]
)

# Labels for top 8
change_labels = (
    v14_base
    .transform_filter("datum.rank <= 8")
    .mark_text(
        align="left",
        dx=8,
        fontSize=11,
        fontWeight="bold",
        color="#333333"
    )
    .encode(
        y=alt.Y(
            "Country/Region/World:N",
            sort=alt.SortField(field="change", order="descending"),
            title=None
        ),
        x="end_prev:Q",
        text=alt.Text("change:Q", format=".2f")
    )
)

# -------------------------
# Final chart
# -------------------------
visual14 = (
    (stems + start_points + end_points + change_labels)
    .add_params(v14_indicator_param, v14_start_param, v14_end_param)
    .properties(
        width=900,
        height=430,
        title={
            "text": "Countries with the fastest rise in prevalence",
            "subtitle": [
                "Top 15 countries ranked by increase between selected years",
                "Grey point = start year, coloured point = end year, label = increase"
            ]
        }
    )
    .configure_title(
        anchor="start",
        fontSize=18,
        subtitleFontSize=12
    )
    .configure_axis(
        labelFontSize=11,
        titleFontSize=12
    )
    .configure_view(
        stroke=None
    )
)

visual14

### Visual 15
Urbanisation Boxplot with Jitter Overlay: How obesity prevalence varies across urbanisation levels

In [ ]:
# =========================
# Visual 15
# Brushed Urbanisation Scatter + Linked Country Sex Split Bars
# Urbanisation, obesity, and gender patterns across selected countries
# =========================

import altair as alt
import pandas as pd
import numpy as np

# -------------------------
# Prepare data
# -------------------------
v15_df = master.copy()

v15_df = v15_df[
    [
        "Country/Region/World",
        "Year",
        "Indicator",
        "Sex",
        "PrevalencePercent",
        "UrbanPopulationPercent",
        "Region"
    ]
].dropna()

# Remove obvious aggregate rows
exclude_values = [
    "World", "Africa", "Asia", "Europe", "North America",
    "South America", "Oceania"
]
v15_df = v15_df[~v15_df["Country/Region/World"].isin(exclude_values)].copy()

# Focus on obesity
indicator_values = sorted(v15_df["Indicator"].unique().tolist())
default_indicator = "Obesity" if "Obesity" in indicator_values else indicator_values[0]
v15_df = v15_df[v15_df["Indicator"] == default_indicator].copy()

# Detect sex labels safely
sex_values = sorted(v15_df["Sex"].unique().tolist())

if "Men" in sex_values and "Women" in sex_values:
    male_label = "Men"
    female_label = "Women"
elif "Male" in sex_values and "Female" in sex_values:
    male_label = "Male"
    female_label = "Female"
else:
    raise ValueError(f"Unexpected Sex labels found: {sex_values}")

v15_df = v15_df[v15_df["Sex"].isin([male_label, female_label])].copy()

# Aggregate
v15_df = (
    v15_df
    .groupby(
        ["Country/Region/World", "Year", "Indicator", "Sex", "Region"],
        as_index=False
    )[["PrevalencePercent", "UrbanPopulationPercent"]]
    .mean()
)

# Country level average across sexes for the top scatter
v15_scatter_df = (
    v15_df
    .groupby(
        ["Country/Region/World", "Year", "Indicator", "Region"],
        as_index=False
    )[["PrevalencePercent", "UrbanPopulationPercent"]]
    .mean()
)

year_options = sorted(v15_df["Year"].unique().tolist())
default_year = 2016 if 2016 in year_options else year_options[-1]
year_min = int(min(year_options))
year_max = int(max(year_options))

# -------------------------
# Safe year parameter
# -------------------------
v15_year = alt.param(
    name="v15_year_final",
    value=default_year,
    bind=alt.binding_range(
        min=year_min,
        max=year_max,
        step=1,
        name="Year: "
    )
)

# -------------------------
# Brush on x axis
# -------------------------
urban_brush = alt.selection_interval(
    name="urban_brush_final",
    encodings=["x"]
)

# -------------------------
# Base charts
# -------------------------
scatter_base = (
    alt.Chart(v15_scatter_df)
    .transform_filter("datum.Year == v15_year_final")
)

bars_base = (
    alt.Chart(v15_df)
    .transform_filter("datum.Year == v15_year_final")
    .transform_filter(urban_brush)
)

# -------------------------
# Urban bands for the top chart
# -------------------------
urban_band_df = pd.DataFrame({
    "band": [
        "Low urbanisation",
        "Moderate urbanisation",
        "High urbanisation",
        "Very high urbanisation"
    ],
    "x1": [0, 25, 50, 75],
    "x2": [25, 50, 75, 100],
    "xmid": [12.5, 37.5, 62.5, 87.5]
})

band_rects = alt.Chart(urban_band_df).mark_rect(opacity=0.08, color="#A0A0A0").encode(
    x=alt.X("x1:Q"),
    x2="x2:Q"
)

band_labels = alt.Chart(urban_band_df).mark_text(
    dy=-145,
    fontSize=11,
    color="#555555",
    fontWeight="bold"
).encode(
    x=alt.X("xmid:Q"),
    text="band:N"
)

# -------------------------
# Top scatter chart
# -------------------------
points = scatter_base.mark_circle(
    size=80,
    opacity=0.8,
    color="#4A4A4A"
).encode(
    x=alt.X(
        "UrbanPopulationPercent:Q",
        title="Urban population (%)",
        scale=alt.Scale(domain=[0, 100])
    ),
    y=alt.Y(
        "PrevalencePercent:Q",
        title="Obesity prevalence (%)"
    ),
    tooltip=[
        alt.Tooltip("Country/Region/World:N", title="Country"),
        alt.Tooltip("Region:N", title="Region"),
        alt.Tooltip("UrbanPopulationPercent:Q", title="Urban population (%)", format=".2f"),
        alt.Tooltip("PrevalencePercent:Q", title="Obesity prevalence (%)", format=".2f"),
        alt.Tooltip("Year:O", title="Year")
    ]
)

highlighted_points = scatter_base.mark_circle(
    size=95,
    opacity=1,
    color="#2F2F2F",
    stroke="white",
    strokeWidth=0.8
).encode(
    x="UrbanPopulationPercent:Q",
    y="PrevalencePercent:Q",
    tooltip=[
        alt.Tooltip("Country/Region/World:N", title="Country"),
        alt.Tooltip("Region:N", title="Region"),
        alt.Tooltip("UrbanPopulationPercent:Q", title="Urban population (%)", format=".2f"),
        alt.Tooltip("PrevalencePercent:Q", title="Obesity prevalence (%)", format=".2f")
    ]
).transform_filter(urban_brush)

trend_line = scatter_base.transform_regression(
    "UrbanPopulationPercent",
    "PrevalencePercent"
).mark_line(
    color="#E45756",
    strokeWidth=3
).encode(
    x="UrbanPopulationPercent:Q",
    y="PrevalencePercent:Q"
)

brush_layer = scatter_base.mark_rect(opacity=0).encode(
    x=alt.X("UrbanPopulationPercent:Q")
).add_params(urban_brush)

top_chart = (
    (band_rects + points + trend_line + highlighted_points + brush_layer + band_labels)
    .properties(
        width=950,
        height=360,
        title={
            "text": "Urbanisation, obesity, and gender patterns across selected countries",
            "subtitle": [
                "Drag across the x axis to select an urbanisation range",
                "The lower chart shows the top 10 countries in that selected range, split by sex"
            ]
        }
    )
)

# -------------------------
# Bottom linked grouped bar chart
# -------------------------
bars_ranked = (
    bars_base
    .transform_joinaggregate(
        country_avg="mean(PrevalencePercent)",
        groupby=["Country/Region/World"]
    )
    .transform_window(
        rank="rank()",
        sort=[alt.SortField("country_avg", order="descending")]
    )
    .transform_filter("datum.rank <= 10")
)

bars = bars_ranked.mark_bar(size=22).encode(
    y=alt.Y(
        "Country/Region/World:N",
        sort=alt.SortField(field="country_avg", order="descending"),
        title=None
    ),
    yOffset=alt.YOffset("Sex:N"),
    x=alt.X(
        "PrevalencePercent:Q",
        title="Obesity prevalence (%)"
    ),
    color=alt.Color(
        "Sex:N",
        scale=alt.Scale(
            domain=[male_label, female_label],
            range=["#4C78A8", "#E45756"]
        ),
        legend=alt.Legend(title="Sex", orient="top-left")
    ),
    tooltip=[
        alt.Tooltip("Country/Region/World:N", title="Country"),
        alt.Tooltip("Sex:N", title="Sex"),
        alt.Tooltip("PrevalencePercent:Q", title="Obesity prevalence (%)", format=".2f"),
        alt.Tooltip("UrbanPopulationPercent:Q", title="Urban population (%)", format=".2f"),
        alt.Tooltip("Region:N", title="Region"),
        alt.Tooltip("Year:O", title="Year")
    ]
)

bar_labels = bars_ranked.mark_text(
    align="left",
    baseline="middle",
    dx=5,
    fontSize=10,
    color="#222222"
).encode(
    y=alt.Y(
        "Country/Region/World:N",
        sort=alt.SortField(field="country_avg", order="descending"),
        title=None
    ),
    yOffset=alt.YOffset("Sex:N"),
    x=alt.X("PrevalencePercent:Q"),
    text=alt.Text("PrevalencePercent:Q", format=".1f")
)

bottom_chart = (
    (bars + bar_labels)
    .properties(
        width=950,
        height=360
    )
)

# -------------------------
# Final chart
# -------------------------
visual15 = (
    alt.vconcat(
        top_chart,
        bottom_chart,
        spacing=24
    )
    .add_params(v15_year)
    .configure_title(
        anchor="start",
        fontSize=18,
        subtitleFontSize=12
    )
    .configure_axis(
        labelFontSize=11,
        titleFontSize=12
    )
    .configure_view(
        stroke=None
    )
)

visual15

### Visual 16
Focus + Context Country Trajectory Chart: Country level prevalence trajectories over time

In [ ]:
# =========================
# Visual 16
# Focus + Context Country Trajectory Chart
# Country level prevalence trajectories over time
# Hover-highlighted background lines with labels and tooltips
# =========================

import altair as alt
import pandas as pd
import numpy as np

# -------------------------
# Prepare data
# -------------------------
v16_df = master.copy()

v16_df = v16_df[
    [
        "Country/Region/World",
        "Year",
        "Indicator",
        "PrevalencePercent",
        "Region"
    ]
].dropna()

exclude_values = [
    "World", "Africa", "Asia", "Europe", "North America",
    "South America", "Oceania"
]
v16_df = v16_df[~v16_df["Country/Region/World"].isin(exclude_values)].copy()

v16_df = (
    v16_df
    .groupby(
        ["Country/Region/World", "Year", "Indicator", "Region"],
        as_index=False
    )["PrevalencePercent"]
    .mean()
)

indicator_options = sorted(v16_df["Indicator"].unique().tolist())
default_indicator = "Obesity" if "Obesity" in indicator_options else indicator_options[0]

region_options = ["All"] + sorted(v16_df["Region"].dropna().unique().tolist())
default_region = "All"

all_countries = sorted(v16_df["Country/Region/World"].unique().tolist())

default_country_1 = "United States" if "United States" in all_countries else all_countries[0]
default_country_2 = "India" if "India" in all_countries else all_countries[min(1, len(all_countries) - 1)]
default_country_3 = "Brazil" if "Brazil" in all_countries else all_countries[min(2, len(all_countries) - 1)]

# -------------------------
# Safe variable params
# -------------------------
v16_indicator = alt.param(
    name="v16_indicator_hover",
    value=default_indicator,
    bind=alt.binding_select(
        options=indicator_options,
        name="Indicator: "
    )
)

v16_region = alt.param(
    name="v16_region_hover",
    value=default_region,
    bind=alt.binding_select(
        options=region_options,
        name="Region: "
    )
)

v16_country_1 = alt.param(
    name="v16_country_1_hover",
    value=default_country_1,
    bind=alt.binding_select(
        options=all_countries,
        name="Country 1: "
    )
)

v16_country_2 = alt.param(
    name="v16_country_2_hover",
    value=default_country_2,
    bind=alt.binding_select(
        options=all_countries,
        name="Country 2: "
    )
)

v16_country_3 = alt.param(
    name="v16_country_3_hover",
    value=default_country_3,
    bind=alt.binding_select(
        options=all_countries,
        name="Country 3: "
    )
)

# -------------------------
# Year brush
# -------------------------
year_brush = alt.selection_interval(
    name="v16_year_brush_hover",
    encodings=["x"]
)

# -------------------------
# Hover selection for background countries
# -------------------------
country_hover = alt.selection_point(
    name="v16_country_hover",
    fields=["Country/Region/World"],
    on="mouseover",
    clear="mouseout",
    empty=False
)

# -------------------------
# Base datasets
# -------------------------
base_all = (
    alt.Chart(v16_df)
    .transform_filter("datum.Indicator == v16_indicator_hover")
)

background_base = (
    base_all
    .transform_filter("v16_region_hover == 'All' || datum.Region == v16_region_hover")
)

highlight_base = (
    base_all
    .transform_filter(
        "datum['Country/Region/World'] == v16_country_1_hover || "
        "datum['Country/Region/World'] == v16_country_2_hover || "
        "datum['Country/Region/World'] == v16_country_3_hover"
    )
)

# -------------------------
# Background grey lines
# -------------------------
background_lines = background_base.mark_line().encode(
    x=alt.X(
        "Year:Q",
        title="Year",
        axis=alt.Axis(format="d", tickMinStep=1)
    ),
    y=alt.Y(
        "PrevalencePercent:Q",
        title="Prevalence (%)"
    ),
    detail="Country/Region/World:N",
    color=alt.condition(
        country_hover,
        alt.value("#7A7A7A"),
        alt.value("#D3D3D3")
    ),
    opacity=alt.condition(
        country_hover,
        alt.value(0.95),
        alt.value(0.28)
    ),
    strokeWidth=alt.condition(
        country_hover,
        alt.value(2.8),
        alt.value(1.0)
    )
).transform_filter(
    year_brush
)

# Invisible hover points to capture hover properly
background_hover_points = background_base.mark_circle(
    size=28,
    opacity=0
).encode(
    x=alt.X("Year:Q", axis=alt.Axis(format="d", tickMinStep=1)),
    y="PrevalencePercent:Q",
    detail="Country/Region/World:N",
    tooltip=[
        alt.Tooltip("Country/Region/World:N", title="Country"),
        alt.Tooltip("Region:N", title="Region"),
        alt.Tooltip("Year:Q", title="Year", format="d"),
        alt.Tooltip("PrevalencePercent:Q", title="Prevalence (%)", format=".2f")
    ]
).transform_filter(
    year_brush
).add_params(
    country_hover
)

# Hover label for the hovered background country at latest visible year
background_hover_label = (
    background_base
    .transform_filter(year_brush)
    .transform_filter(country_hover)
    .transform_joinaggregate(
        max_year="max(Year)",
        groupby=["Country/Region/World"]
    )
    .transform_filter("datum.Year == datum.max_year")
    .mark_text(
        align="left",
        baseline="middle",
        dx=6,
        fontSize=11,
        fontWeight="bold",
        color="#4F4F4F"
    )
    .encode(
        x=alt.X("Year:Q", axis=alt.Axis(format="d", tickMinStep=1)),
        y="PrevalencePercent:Q",
        text="Country/Region/World:N"
    )
)

# -------------------------
# Highlighted selected countries
# -------------------------
highlight_lines = highlight_base.mark_line(
    strokeWidth=3.2
).encode(
    x=alt.X(
        "Year:Q",
        title="Year",
        axis=alt.Axis(format="d", tickMinStep=1)
    ),
    y=alt.Y(
        "PrevalencePercent:Q",
        title="Prevalence (%)"
    ),
    color=alt.Color(
        "Country/Region/World:N",
        legend=alt.Legend(title="Highlighted countries", orient="top-left")
    ),
    detail="Country/Region/World:N",
    tooltip=[
        alt.Tooltip("Country/Region/World:N", title="Country"),
        alt.Tooltip("Region:N", title="Region"),
        alt.Tooltip("Year:Q", title="Year", format="d"),
        alt.Tooltip("PrevalencePercent:Q", title="Prevalence (%)", format=".2f")
    ]
).transform_filter(
    year_brush
)

highlight_points = highlight_base.mark_circle(
    size=70,
    opacity=0.95
).encode(
    x=alt.X("Year:Q", axis=alt.Axis(format="d", tickMinStep=1)),
    y="PrevalencePercent:Q",
    color=alt.Color("Country/Region/World:N", legend=None),
    tooltip=[
        alt.Tooltip("Country/Region/World:N", title="Country"),
        alt.Tooltip("Region:N", title="Region"),
        alt.Tooltip("Year:Q", title="Year", format="d"),
        alt.Tooltip("PrevalencePercent:Q", title="Prevalence (%)", format=".2f")
    ]
).transform_filter(
    year_brush
)

selected_end_labels = (
    highlight_base
    .transform_filter(year_brush)
    .transform_joinaggregate(
        max_year="max(Year)",
        groupby=["Country/Region/World"]
    )
    .transform_filter("datum.Year == datum.max_year")
    .mark_text(
        align="left",
        baseline="middle",
        dx=6,
        fontSize=11,
        fontWeight="bold"
    )
    .encode(
        x=alt.X("Year:Q", axis=alt.Axis(format="d", tickMinStep=1)),
        y="PrevalencePercent:Q",
        text="Country/Region/World:N",
        color=alt.Color("Country/Region/World:N", legend=None)
    )
)

# -------------------------
# Main chart
# -------------------------
main_chart = (
    (
        background_lines
        + background_hover_points
        + background_hover_label
        + highlight_lines
        + highlight_points
        + selected_end_labels
    )
    .properties(
        width=950,
        height=400,
        title={
            "text": "Country level prevalence trajectories over time",
            "subtitle": [
                "Hover over grey lines to identify countries and view details",
                "Grey lines follow the selected region, while highlighted countries always remain visible"
            ]
        }
    )
)

# -------------------------
# Context chart
# -------------------------
context_background = background_base.mark_line(
    color="#BEBEBE",
    opacity=0.38,
    strokeWidth=1
).encode(
    x=alt.X(
        "Year:Q",
        title="Year",
        axis=alt.Axis(format="d", tickMinStep=1)
    ),
    y=alt.Y(
        "mean(PrevalencePercent):Q",
        title="Average prevalence (%)"
    ),
    detail="Country/Region/World:N"
)

context_highlight = highlight_base.mark_line(
    strokeWidth=2.2
).encode(
    x=alt.X(
        "Year:Q",
        title="Year",
        axis=alt.Axis(format="d", tickMinStep=1)
    ),
    y=alt.Y(
        "PrevalencePercent:Q",
        title="Average prevalence (%)"
    ),
    color=alt.Color("Country/Region/World:N", legend=None),
    detail="Country/Region/World:N"
)

context_chart = (
    (context_background + context_highlight)
    .add_params(year_brush)
    .properties(
        width=950,
        height=90
    )
)

# -------------------------
# Final chart
# -------------------------
visual16 = (
    alt.vconcat(
        main_chart,
        context_chart,
        spacing=16
    )
    .add_params(
        v16_indicator,
        v16_region,
        v16_country_1,
        v16_country_2,
        v16_country_3
    )
    .configure_title(
        anchor="start",
        fontSize=18,
        subtitleFontSize=12
    )
    .configure_axis(
        labelFontSize=11,
        titleFontSize=12
    )
    .configure_view(
        stroke=None
    )
)

visual16

### Visual 17
Health Expenditure Quadrant Bubble Chart: Health expenditure per capita versus NCD prevalence

In [ ]:
# =========================
# Visual 17
# Health expenditure, prevalence, and sex differences
# FINAL WORKING VERSION
# =========================

import altair as alt
import pandas as pd
import numpy as np

alt.data_transformers.disable_max_rows()

# -------------------------
# Prepare data
# -------------------------
v17 = master.copy()

needed_cols = [
    "Country/Region/World",
    "Sex",
    "Year",
    "Indicator",
    "PrevalencePercent",
    "HealthExpenditurePerCapita",
    "Age65PlusPercent",
    "Region"
]

v17 = v17[needed_cols].dropna().copy()

exclude_values = [
    "World", "Africa", "Asia", "Europe", "North America",
    "South America", "Oceania"
]
v17 = v17[~v17["Country/Region/World"].isin(exclude_values)].copy()

v17 = v17[v17["Sex"].isin(["Men", "Women"])].copy()

# -------------------------
# One shared country-level table
# -------------------------
country_level = (
    v17.pivot_table(
        index=["Country/Region/World", "Year", "Indicator", "Region"],
        columns="Sex",
        values="PrevalencePercent",
        aggfunc="mean"
    )
    .reset_index()
)

country_level.columns.name = None

extra_cols = (
    v17.groupby(
        ["Country/Region/World", "Year", "Indicator", "Region"],
        as_index=False
    )[["HealthExpenditurePerCapita", "Age65PlusPercent"]]
    .mean()
)

country_level = country_level.merge(
    extra_cols,
    on=["Country/Region/World", "Year", "Indicator", "Region"],
    how="left"
)

country_level = country_level.dropna(subset=["Men", "Women"]).copy()

country_level["AvgPrevalence"] = (
    country_level["Men"] + country_level["Women"]
) / 2

# -------------------------
# Controls
# -------------------------
indicator_options = sorted(country_level["Indicator"].dropna().unique().tolist())
default_indicator = "Obesity" if "Obesity" in indicator_options else indicator_options[0]

year_options = sorted(country_level["Year"].dropna().unique().tolist())
default_year = 2016 if 2016 in year_options else year_options[-1]

v17_indicator = alt.param(
    name="v17_indicator",
    value=default_indicator,
    bind=alt.binding_select(
        options=indicator_options,
        name="Indicator: "
    )
)

v17_year = alt.param(
    name="v17_year",
    value=int(default_year),
    bind=alt.binding_range(
        min=int(min(year_options)),
        max=int(max(year_options)),
        step=1,
        name="Year: "
    )
)

# IMPORTANT:
# attach brush to a real layer using the country dataset
exp_brush = alt.selection_interval(
    name="exp_brush",
    encodings=["x"],
    empty=True
)

bubble_hover = alt.selection_point(
    name="bubble_hover",
    fields=["Country/Region/World"],
    on="mouseover",
    clear="mouseout",
    empty=False
)

# -------------------------
# Shared base
# -------------------------
base = (
    alt.Chart(country_level)
    .transform_filter("datum.Indicator == v17_indicator")
    .transform_filter("datum.Year == v17_year")
)

# -------------------------
# Median lines
# -------------------------
median_df = (
    country_level.groupby(["Indicator", "Year"], as_index=False)
    .agg(
        median_exp=("HealthExpenditurePerCapita", "median"),
        median_prev=("AvgPrevalence", "median")
    )
)

median_base = (
    alt.Chart(median_df)
    .transform_filter("datum.Indicator == v17_indicator")
    .transform_filter("datum.Year == v17_year")
)

vertical_rule = median_base.mark_rule(
    strokeDash=[6, 4],
    color="#666666",
    strokeWidth=1.4,
    opacity=0.8
).encode(
    x="median_exp:Q"
)

horizontal_rule = median_base.mark_rule(
    strokeDash=[6, 4],
    color="#666666",
    strokeWidth=1.4,
    opacity=0.8
).encode(
    y="median_prev:Q"
)

# -------------------------
# Invisible selector layer
# This is what makes the brush drive everything correctly
# -------------------------
selector_layer = base.mark_point(
    opacity=0
).encode(
    x=alt.X("HealthExpenditurePerCapita:Q"),
    y=alt.Y("AvgPrevalence:Q")
).add_params(
    exp_brush
)

# -------------------------
# Bubble chart
# -------------------------
bubbles = base.mark_circle(
    filled=True
).encode(
    x=alt.X(
        "HealthExpenditurePerCapita:Q",
        title="Health expenditure per capita"
    ),
    y=alt.Y(
        "AvgPrevalence:Q",
        title="Prevalence (%)"
    ),
    size=alt.Size(
        "Age65PlusPercent:Q",
        title="Age 65+ (%)",
        scale=alt.Scale(range=[50, 1100])
    ),
    color=alt.Color(
        "Region:N",
        scale=alt.Scale(scheme="tableau10"),
        legend=alt.Legend(title="Region")
    ),
    opacity=alt.value(0.85),
    stroke=alt.condition(
        exp_brush,
        alt.value("black"),
        alt.value("white")
    ),
    strokeWidth=alt.condition(
        exp_brush,
        alt.value(1.6),
        alt.value(0.8)
    ),
    tooltip=[
        alt.Tooltip("Country/Region/World:N", title="Country"),
        alt.Tooltip("Region:N", title="Region"),
        alt.Tooltip("HealthExpenditurePerCapita:Q", title="Health expenditure per capita", format=",.2f"),
        alt.Tooltip("AvgPrevalence:Q", title="Average prevalence (%)", format=".2f"),
        alt.Tooltip("Men:Q", title="Men prevalence (%)", format=".2f"),
        alt.Tooltip("Women:Q", title="Women prevalence (%)", format=".2f"),
        alt.Tooltip("Age65PlusPercent:Q", title="Age 65+ (%)", format=".2f")
    ]
)

hover_points = base.mark_circle(
    size=160,
    opacity=0
).encode(
    x="HealthExpenditurePerCapita:Q",
    y="AvgPrevalence:Q"
).add_params(
    bubble_hover
)

hover_labels = (
    base
    .transform_filter(bubble_hover)
    .mark_text(
        align="left",
        dx=6,
        dy=-6,
        fontSize=11,
        fontWeight="bold",
        color="#333333"
    )
    .encode(
        x="HealthExpenditurePerCapita:Q",
        y="AvgPrevalence:Q",
        text="Country/Region/World:N"
    )
)

bubble_chart = (
    vertical_rule
    + horizontal_rule
    + selector_layer
    + bubbles
    + hover_points
    + hover_labels
).properties(
    width=720,
    height=430,
    title={
        "text": "Health expenditure, prevalence, and sex differences",
        "subtitle": [
            "Drag across the x axis to select a health expenditure range",
            "The donut chart updates to show average men versus women prevalence for countries in the selected range"
        ]
    }
)

# -------------------------
# Donut chart
# SAME brushed source
# -------------------------
donut_base = (
    base
    .transform_filter(exp_brush)
    .transform_fold(
        ["Men", "Women"],
        as_=["Sex", "PrevalencePercent"]
    )
    .transform_aggregate(
        avg_prev="mean(PrevalencePercent)",
        groupby=["Sex"]
    )
    .transform_joinaggregate(
        total_prev="sum(avg_prev)"
    )
    .transform_calculate(
        pct="datum.avg_prev / datum.total_prev"
    )
)

donut = donut_base.mark_arc(
    innerRadius=78,
    outerRadius=140
).encode(
    theta=alt.Theta("avg_prev:Q", stack=True),
    color=alt.Color(
        "Sex:N",
        scale=alt.Scale(
            domain=["Men", "Women"],
            range=["#4C78A8", "#E45756"]
        ),
        legend=alt.Legend(title="Sex", orient="top")
    ),
    tooltip=[
        alt.Tooltip("Sex:N", title="Sex"),
        alt.Tooltip("avg_prev:Q", title="Average prevalence (%)", format=".2f"),
        alt.Tooltip("pct:Q", title="Share of total", format=".1%")
    ]
)

# -------------------------
# Country count
# SAME brushed source
# -------------------------
count_base = (
    base
    .transform_filter(exp_brush)
    .transform_aggregate(
        country_count="count()"
    )
)

center_count = count_base.mark_text(
    fontSize=28,
    fontWeight="bold",
    color="#333333"
).encode(
    x=alt.value(160),
    y=alt.value(190),
    text=alt.Text("country_count:Q", format=".0f")
)

center_label = alt.Chart(
    pd.DataFrame({"label": ["countries in range"]})
).mark_text(
    fontSize=12,
    color="#666666"
).encode(
    x=alt.value(160),
    y=alt.value(214),
    text="label:N"
)

donut_chart = (
    donut + center_count + center_label
).properties(
    width=320,
    height=430
)

# -------------------------
# Final chart
# -------------------------
visual17 = (
    alt.hconcat(
        bubble_chart,
        donut_chart,
        spacing=24
    )
    .add_params(v17_indicator, v17_year)
    .resolve_scale(
        color="independent",
        size="independent"
    )
    .configure_title(
        anchor="start",
        fontSize=18,
        subtitleFontSize=12
    )
    .configure_axis(
        labelFontSize=11,
        titleFontSize=12
    )
    .configure_view(
        stroke=None
    )
)

visual17

### Mini Dashboard

In [ ]:
# =========================
# Final 6 Box Dashboard
# Equal grid + more varied visuals
# =========================

import altair as alt
import pandas as pd
import numpy as np

alt.data_transformers.disable_max_rows()

# =========================
# Detect GDP column automatically
# =========================
possible_gdp_cols = [
    "GDPPerCapita",
    "GDP per capita",
    "GdpPerCapita",
    "GDPPerCapitaCurrentUS",
    "GDP per capita (current US$)",
    "GDPPerCapitaCurrentUSDollar",
    "GDPCapita",
    "GDP"
]

gdp_col = None
for col in possible_gdp_cols:
    if col in master.columns:
        gdp_col = col
        break

if gdp_col is None:
    gdp_matches = [col for col in master.columns if "gdp" in col.lower()]
    raise ValueError(f"No GDP column found. GDP-like columns available: {gdp_matches}")

# =========================
# Prepare data
# =========================
df = master.copy()

needed_cols = [
    "Country/Region/World",
    "Sex",
    "Year",
    "Indicator",
    "PrevalencePercent",
    "UrbanPopulationPercent",
    "HealthExpenditurePerCapita",
    gdp_col,
    "Age65PlusPercent",
    "Region",
    "IncomeGroup"
]

df = df[needed_cols].dropna().copy()

exclude_values = [
    "World", "Africa", "Asia", "Europe",
    "North America", "South America", "Oceania"
]
df = df[~df["Country/Region/World"].isin(exclude_values)].copy()

sex_values = sorted(df["Sex"].dropna().unique().tolist())

if "Men" in sex_values and "Women" in sex_values:
    male_label = "Men"
    female_label = "Women"
elif "Male" in sex_values and "Female" in sex_values:
    male_label = "Male"
    female_label = "Female"
else:
    raise ValueError(f"Unexpected Sex labels found: {sex_values}")

df = df[df["Sex"].isin([male_label, female_label])].copy()

# =========================
# Aggregated tables
# =========================
sex_level = (
    df.groupby(
        ["Country/Region/World", "Year", "Indicator", "Sex", "Region", "IncomeGroup"],
        as_index=False
    )[[
        "PrevalencePercent",
        "UrbanPopulationPercent",
        "HealthExpenditurePerCapita",
        gdp_col,
        "Age65PlusPercent"
    ]]
    .mean()
)

country_level = (
    sex_level.pivot_table(
        index=["Country/Region/World", "Year", "Indicator", "Region", "IncomeGroup"],
        columns="Sex",
        values="PrevalencePercent",
        aggfunc="mean"
    )
    .reset_index()
)

country_level.columns.name = None

extra_cols = (
    sex_level.groupby(
        ["Country/Region/World", "Year", "Indicator", "Region", "IncomeGroup"],
        as_index=False
    )[[
        "UrbanPopulationPercent",
        "HealthExpenditurePerCapita",
        gdp_col,
        "Age65PlusPercent"
    ]]
    .mean()
)

country_level = country_level.merge(
    extra_cols,
    on=["Country/Region/World", "Year", "Indicator", "Region", "IncomeGroup"],
    how="left"
)

country_level = country_level.dropna(subset=[male_label, female_label]).copy()
country_level["AvgPrevalence"] = (
    country_level[male_label] + country_level[female_label]
) / 2
country_level["SexGap"] = (
    country_level[female_label] - country_level[male_label]
)

region_df = (
    country_level.groupby(["Region", "Year", "Indicator"], as_index=False)
    [[
        "AvgPrevalence",
        "UrbanPopulationPercent",
        "HealthExpenditurePerCapita",
        gdp_col
    ]]
    .mean()
)

income_df = (
    country_level.groupby(["IncomeGroup", "Year", "Indicator"], as_index=False)
    [["AvgPrevalence"]]
    .mean()
)

region_sex_df = (
    sex_level.groupby(["Region", "Year", "Indicator", "Sex"], as_index=False)
    [["PrevalencePercent"]]
    .mean()
)

# =========================
# Controls
# =========================
indicator_options = sorted(country_level["Indicator"].dropna().unique().tolist())
default_indicator = "Obesity" if "Obesity" in indicator_options else indicator_options[0]

year_options = sorted(country_level["Year"].dropna().unique().tolist())
default_year = 2016 if 2016 in year_options else year_options[-1]

indicator_param = alt.param(
    name="dashboard_indicator_boxes",
    value=default_indicator,
    bind=alt.binding_select(
        options=indicator_options,
        name="Indicator: "
    )
)

year_param = alt.param(
    name="dashboard_year_boxes",
    value=int(default_year),
    bind=alt.binding_range(
        min=int(min(year_options)),
        max=int(max(year_options)),
        step=1,
        name="Year: "
    )
)

hover_urban = alt.selection_point(
    name="hover_urban_boxes",
    fields=["Country/Region/World"],
    on="mouseover",
    clear="mouseout",
    empty=False
)

# =========================
# Shared bases
# =========================
country_base = (
    alt.Chart(country_level)
    .transform_filter("datum.Indicator == dashboard_indicator_boxes")
    .transform_filter("datum.Year == dashboard_year_boxes")
)

region_base = (
    alt.Chart(region_df)
    .transform_filter("datum.Indicator == dashboard_indicator_boxes")
    .transform_filter("datum.Year == dashboard_year_boxes")
)

income_base = (
    alt.Chart(income_df)
    .transform_filter("datum.Indicator == dashboard_indicator_boxes")
)

region_sex_base = (
    alt.Chart(region_sex_df)
    .transform_filter("datum.Indicator == dashboard_indicator_boxes")
    .transform_filter("datum.Year == dashboard_year_boxes")
)

# =========================
# Equal box sizes
# =========================
box_w = 320
box_h = 230

# =========================
# Panel 1
# Regional ranking
# =========================
panel1 = (
    region_base
    .mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4)
    .encode(
        x=alt.X("AvgPrevalence:Q", title="Average prevalence (%)"),
        y=alt.Y("Region:N", sort="-x", title=None),
        color=alt.Color("Region:N", legend=None),
        tooltip=[
            alt.Tooltip("Region:N", title="Region"),
            alt.Tooltip("AvgPrevalence:Q", title="Average prevalence (%)", format=".2f"),
            alt.Tooltip("UrbanPopulationPercent:Q", title="Urban population (%)", format=".2f"),
            alt.Tooltip("HealthExpenditurePerCapita:Q", title="Health expenditure per capita", format=",.2f"),
            alt.Tooltip(f"{gdp_col}:Q", title="GDP per capita", format=",.2f")
        ]
    )
    .properties(
        width=box_w,
        height=box_h,
        title="Regional prevalence ranking"
    )
)

# =========================
# Panel 2
# Income group heatmap
# =========================
panel2 = (
    income_base
    .mark_rect(cornerRadius=2)
    .encode(
        x=alt.X("Year:O", title="Year", axis=alt.Axis(labelAngle=-35)),
        y=alt.Y(
            "IncomeGroup:N",
            title=None,
            sort=[
                "Low income",
                "Lower middle income",
                "Upper middle income",
                "High income"
            ]
        ),
        color=alt.Color("AvgPrevalence:Q", title="Avg prevalence (%)"),
        tooltip=[
            alt.Tooltip("IncomeGroup:N", title="Income group"),
            alt.Tooltip("Year:O", title="Year"),
            alt.Tooltip("AvgPrevalence:Q", title="Average prevalence (%)", format=".2f")
        ]
    )
    .properties(
        width=box_w,
        height=box_h,
        title="Income group trends"
    )
)

# =========================
# Panel 3
# Male versus female by region
# =========================
panel3 = (
    region_sex_base
    .mark_bar(cornerRadiusTopLeft=3, cornerRadiusTopRight=3)
    .encode(
        x=alt.X("Region:N", title=None, axis=alt.Axis(labelAngle=-25)),
        y=alt.Y("PrevalencePercent:Q", title="Average prevalence (%)"),
        xOffset="Sex:N",
        color=alt.Color(
            "Sex:N",
            scale=alt.Scale(
                domain=[male_label, female_label],
                range=["#4C78A8", "#E45756"]
            ),
            legend=alt.Legend(title="Sex", orient="right")
        ),
        tooltip=[
            alt.Tooltip("Region:N", title="Region"),
            alt.Tooltip("Sex:N", title="Sex"),
            alt.Tooltip("PrevalencePercent:Q", title="Average prevalence (%)", format=".2f")
        ]
    )
    .properties(
        width=box_w,
        height=box_h,
        title="Male versus female by region"
    )
)

# =========================
# Panel 4
# Urbanisation scatter
# =========================
urban_regression = (
    country_base
    .transform_regression("UrbanPopulationPercent", "AvgPrevalence")
    .mark_line(strokeDash=[6, 4], color="#666666", strokeWidth=2)
    .encode(
        x="UrbanPopulationPercent:Q",
        y="AvgPrevalence:Q"
    )
)

urban_points = (
    country_base
    .mark_circle(opacity=0.82)
    .encode(
        x=alt.X("UrbanPopulationPercent:Q", title="Urban population (%)"),
        y=alt.Y("AvgPrevalence:Q", title="Average prevalence (%)"),
        size=alt.Size(
            "Age65PlusPercent:Q",
            scale=alt.Scale(range=[35, 500]),
            legend=None
        ),
        color=alt.Color("Region:N", legend=None),
        tooltip=[
            alt.Tooltip("Country/Region/World:N", title="Country"),
            alt.Tooltip("Region:N", title="Region"),
            alt.Tooltip("IncomeGroup:N", title="Income group"),
            alt.Tooltip("UrbanPopulationPercent:Q", title="Urban population (%)", format=".2f"),
            alt.Tooltip("AvgPrevalence:Q", title="Average prevalence (%)", format=".2f"),
            alt.Tooltip("Age65PlusPercent:Q", title="Age 65+ (%)", format=".2f")
        ]
    )
)

urban_hover_layer = (
    country_base
    .mark_circle(size=140, opacity=0)
    .encode(
        x="UrbanPopulationPercent:Q",
        y="AvgPrevalence:Q"
    )
    .add_params(hover_urban)
)

urban_labels = (
    country_base
    .transform_filter(hover_urban)
    .mark_text(
        align="left",
        dx=6,
        dy=-6,
        fontSize=10,
        fontWeight="bold",
        color="#333333"
    )
    .encode(
        x="UrbanPopulationPercent:Q",
        y="AvgPrevalence:Q",
        text="Country/Region/World:N"
    )
)

panel4 = (
    (urban_regression + urban_points + urban_hover_layer + urban_labels)
    .properties(
        width=box_w,
        height=box_h,
        title="Urbanisation and prevalence, size = age 65+"
    )
)

# =========================
# Panel 5
# Health expenditure boxplot by band
# =========================
exp_band_base = (
    country_base
    .transform_calculate(
        ExpBand="""
        datum.HealthExpenditurePerCapita < 500 ? 'Below 500' :
        datum.HealthExpenditurePerCapita < 1500 ? '500 to 1,499' :
        datum.HealthExpenditurePerCapita < 3000 ? '1,500 to 2,999' :
        datum.HealthExpenditurePerCapita < 5000 ? '3,000 to 4,999' :
        '5,000+'
        """
    )
)

exp_box = (
    exp_band_base
    .mark_boxplot(extent="min-max", size=32)
    .encode(
        x=alt.X(
            "ExpBand:N",
            sort=["Below 500", "500 to 1,499", "1,500 to 2,999", "3,000 to 4,999", "5,000+"],
            title="Health expenditure bands (current US$)"
        ),
        y=alt.Y("AvgPrevalence:Q", title="Average prevalence (%)"),
        color=alt.Color("ExpBand:N", legend=None),
        tooltip=[
            alt.Tooltip("ExpBand:N", title="Expenditure band"),
            alt.Tooltip("AvgPrevalence:Q", title="Average prevalence (%)", format=".2f")
        ]
    )
)

panel5 = (
    exp_box.properties(
        width=box_w,
        height=box_h,
        title="Prevalence by health expenditure band"
    )
)

# =========================
# Panel 6
# GDP density heatmap
# =========================
panel6 = (
    country_base
    .mark_rect()
    .encode(
        x=alt.X(
            f"{gdp_col}:Q",
            bin=alt.Bin(maxbins=18),
            title="GDP per capita"
        ),
        y=alt.Y(
            "AvgPrevalence:Q",
            bin=alt.Bin(maxbins=18),
            title="Average prevalence (%)"
        ),
        color=alt.Color("count():Q", title="Country count"),
        tooltip=[
            alt.Tooltip("count():Q", title="Countries in bin")
        ]
    )
    .properties(
        width=box_w,
        height=box_h,
        title="GDP and prevalence density"
    )
)

# =========================
# Final dashboard
# =========================
dashboard = (
    alt.vconcat(
        alt.hconcat(panel1, panel2, panel3, spacing=25),
        alt.hconcat(panel4, panel5, panel6, spacing=25),
        spacing=26
    )
    .add_params(indicator_param, year_param)
    .resolve_scale(
        color="independent",
        size="independent"
    )
    .configure_title(
        anchor="start",
        fontSize=20,
        subtitleFontSize=12
    )
    .configure_axis(
        labelFontSize=11,
        titleFontSize=12
    )
    .configure_legend(
        labelFontSize=10,
        titleFontSize=11
    )
    .configure_view(
        stroke="#D9D9D9"
    )
    .properties(
        title={
            "text": "Dashboard: Global health risk indicators",
            "subtitle": [
                "Six aligned panels covering region, income, sex, urbanisation, health expenditure, and GDP"
            ]
        }
    )
)

dashboard